# CIC-DDoS2019 LightGBM CPU baseline

Production: full natural-distribution train split and exactly 100 boosting iterations. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import json
import os
import subprocess
import sys
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPGtz20aS3/UrsKhSBfBSkGwnW1vM8eoSx3Zcaycu23t3WzoVCiKGElYgwMUAthUt//t1z/sFEqSdu/twrkpEYGZ6evrdPTOI4/hZV9DbM1qsSPS6urntX/74JvqxbWlPumh5S5Z3m7ZqehoVTRl9JF21qkgZFX27rpbR+6cRvW+WWRzHJyerrl1Heb4a+qEjeR5V603b9TCuafuir9qGnpyId0v6Uf68hdnr6lo+/p22jfxdtzc3VXMjH1sqf9Hboa9q+dRXayJ/D0NVckTKoifYItGQzzPW/7e2IbzfpuhxftntLTzyhv5+A5PL9z8097PoTbHBd7PoPfnHQJolOTk5ef3ry5fP30ULiW12Q/rX8JN0SZ43xRookUK3kqyioV/mTfspSaOzf41o381PIvjXEaBXo/DLsIdEMYMhaVbRdtV266JPJCR6Wzz57k/5qqpJgguYM7xnwK+hucuv73tC5xFwDdB6fPHk2+gR+2PPW1Y3hGIPwYGMA4U5sPVT1d8y2mTthjRJ3F3HaVRQ6NyUNeEQWL9bwCG6rtvlXTRfiOasI0WZGMikeoCeOhs2uOiEDU4tWvD2W/KZ/1LrXhZN21TLos4RaVj6fd0W5Ry5Yy8OmNOWIKcLJlBZOaw3VHafQStFES3osqoWL4qaglRQ4HJ+R+7p4kM34DPZFB2IeUcXSTyLZ1E8j9M044CTeOhXZ3+OLawdOgoU0sAyuPbkiFqOqFkLmUUl9K0apjKcsWxtv4A8CM7p9gyQJE2fre/Kqkv4g1wB+VzRPm/v2CPHtCcozUV3D4QxoSC3czqsVtXnxHzPX0V/jOKsX29iQzQUJCEfn+IZJzqowEJSJyQwih+aHbxLmC1VU8KSFk9cBqUKoBC5T10FshT/VxN7Tat6AGHRr1uardBsJbIdRLhpk5T3gNaObOpiSRK1SIsnDhdvgcptd88ZKR7mykJcCptxCaI5Q/ZeXc0YDXJLb+lH49lhtyctNUwiZ0oNaBx/CesLREOBMOVCvQwKBXiFukRzB4YH8cP1XgGoyyve3HagJsu2K4GjkaSS4gg2A2uxjfey7UW1Yq3gR7CHMZXVy0YjA7qTpkxg4IFiO4sa8qmuGrKIR2weilrH6ZT9VC37/2AvEinHGomF/pk6w7nA3oKhhJHhxq79RBWffw+hliyVEv2xqCs0yVKmJ4rzcuhQuHJEWVgt8D2OFKtGilIB7Qnn9GWsWuKrNCwoXIbI5w1Z9syoMw3oiuaGJI/RRvSJh0MKAvpYrB7kx5j+DwsFSvO0KypKon8v6oE877q2SyzRWsUCkwyVLVoP4DiXbdMXgCP+rZqhHWg0NBWQyZzrcZY9eKhtv49iB3x7TUn3EdaGArF40BAu599dbSOYqLbenn03v9pqIMjAZV1QChHZe8BT0BzCMojQirLYoLwOFOMZrefDhptfHtQtGWRgZrsBydDBHn/PIzzufFYQ5FVN1ed5Qkm9wiHNqrqZR55soG4V1zUp8xaAdVVJ5tF129bRP5lgACPxjyMozIYxiOAMkNk4IuFvLmMBEERFdUYkMvEelVKPBb67CEQV5XMTcC8ctNvFgdyB6FcBTDj5ctnsYbQuPkNj31UEJR6elKgKAEYHGOzN2oNPuS4oySmoQ1MikBXMqMf7XTwcroflHcEYD/SfNB+rDjgJ8WkCHFJgeJ8cmmE4hDlxmkFztUkcWJuOoNnfCYv3CcESf+Nzwz135AZEazdE3mcndsBl9Ay8p+0SpsywKur6ulje5ROmEqwRQMUPsFgoUR61aHXTgImBaHbZa52wZV8NUv2ZQfYxBkXO3757/v7Vy1+e/5Q/+/WXF69e5m9/+PBzPEoUG6RNGOYMMdZI7F4p94uTIrnwemXIjcohfFFqDdH6mSvxRH7YYKRcmiLtjFYCGRpNn4p2ABCSPUEjU0vQ5pnPf1j4uPqr9xxH/FaRQg5nJjQSYMtqtSIdjViSCVz98a/P/vL8wwhqYo0KNfFso8ZffglqAqyLGgjci1f/6aBmGxaPQich0eB88lC2VSZf1hX4SZYEjWiJJIu09kiXBJXfRAr0Ub3i8zjpJ0RjtLhB4LEoYNyCmlW/cVoIg041b9g0mhxBRkk3MMaFd0ODST3ng0DApisvJGSfiq4BzUviU/p9BJlxUVtVmI6sMeiQbnAWBWE5LpHlUtx9/9umAw3v+nvlzDnVmQIxTwzUn3sEF7yR3tNeZW+G8mokr5tct3371Gokn5dk00evWDujBxoXeDuJdDEDiHgov/zpljRRgJHQR5Ip5SINk/hkkktbcFwzQQ+wIEBd4RMwgl8Yxt/0YSz3NyHpOAlo/XeIMzFrFaFSR2pA7iPEQWCT7JIFh8abmXPhP7M66DnZrKv4wZDy7fmDHLSNXQvCwh3ZrDHshsbEDp6qkuE2AVW9cHOdgBMHY6Aj1/DN+TfpFpahA0kWw4jZUS5F+sAQWArHrMs7lmhCuAtRCwrFPHrOZAqZHrYbmFUUPYa+LINUmYMXqGHOMEG4xeIlhkkaknCN1Jh86yVEC084hfpJtP/FQ9aHx8laF5jDj0WRj6In0aNHUSLhnsGKg4BcewQadkoVOqfl+WkZrSAvAQ1MTmn6fcQmYyXTJjrNHq9obPB0Jkf6RJ9hqZUksP40kyXTGV9HGDNWJqU1IZvE6QUZEMHCsaYrGAF0BpY0cLOiO1nyCAkc7emIOfSEQVg5OezkQCMXMnASlmXjUIB1iAWs6EGGKTrrkG0T4qmw0uvT0V7ZfmpYjMaVD1RXK7qVyLFCzj8dsy8I7YR+IWJ/xPBDyqPqymNaiQNKysM2ZS9VvUYm7pC10r5oliRhoBhCqRvms/U+xAAgnkeiX4y5tHzcusRhb4VPpLg/sVyT/rYtNa1gKfmqHZoSRdOwMYxEmAqaxpBuIC3HdQIKIOkdjgEUZANfoM8g3sqp8U48vQEIZdEXBlHinz98ePseAA/0WVsS4PliEX178S3GOhj9OtQwoTKpMkExACxp0OUDVNmHGABiwy/t+2F5+xdyzx/6F0iDeKulqL1m1p4VEWlIfmziiPzMjEmC7ONRSiAC9qXLHg8+t1MmLyTilkwZKLGBo9bdxsdhM5+Luy8PQKxxYDI4C1j+9XVZzBUYYXVwswDwZXhdcvm94htW7dAvHj+5SG1Q3qoU7ykTlhy3IISsHLHOjNknUIIu5/AcRycAYBF53ESalGJLyrn8gHRZVODBU2b0SH5kAf3CCO5nEcjlAvm5H5NJblgFuJa2p4cQC2mklUOUhXjRXigHbYduSWSxf1U1RZ1/dYWh1W8omFhh4vMxGWCVlxzbNMFYPkGHNYqxsZHIR6VfrIGOkecEMU28IsAhSqmTthUYJcMZckqf8Wl0wMFc5oOaahuHgqs9qZt20SB1O1K3QLg02m+H0gm/v/Cswv6AlNVzBNt3b9jasSLkKEOfq5pw4oc87j+xvUAJuUsu0tFuTgiSwTTSrFkzon1DZ7eQmyfK2P354kJkCj7saWY4YI7tuWejQ6x+4W4h1k4zm9Z2G/MQU1biOJW9y5DGVbHAdS02E8adTHix9gKmLBmDOTBOzkC+A0ZlbNL0gOHZa9Lc9LdgMs4epylWu9CA7dLPV796FS690YG1ATSO5oaGSF1iH03cAMkPETGDMWwsszlTmYJ6MSqFglkGVGCUINiCNXKysQ7iPbezmp9/urgIi+8UJlu0mMJjbm+PFemDKOeJs6LQoaKskT5AjI1Bv5sIszkOFN/dx3r8Zd/wAvwBrIIRh3EK6SIYpcdesUSOFGtxJMGy+9O5BgAnySUEA/yQEjtAIEfijmq+5BwT55WQ1Avz0FSYZyAFDN64oxw95OSA8Q87sfK+iNGOlZj3P/9wBmyfKDSh8PmgStYBsdS+yMlYDaxDhHUcc1kQL7q+WhXLXlTDsSoVndI5/BfLYDtUVDogGGN0rANhUDD6MsO3rCQ1AYYLgedP467Wg+WSfD6pOPesHeqShc98QiM+1n6QZ1VMFxilAnG4ds544AZjbSOExuM+Zw94xDLD/33L0tXPRmTtESdIrf22RuRQmJXssTFm5miMGnepbK+QJzqzyEosraWPR4bPP/dd8UN3QxcPsarUzKOHmNtZ+ClVd7udHe95j4gP3ajw6PTaokQ6AdlgZHcpXaHwhFfKC2LVyokCmesMFb4EVQ+2iav4gxn/BcwgTxCttW4DpnE/3VmQJGsak6XVGDUurQEOjfZFzildHe/2DOZ9zxQABJjDj+e2JsRY+5vbCrEdhyjZ9hMY/iVu9yziZ7++/Vv8BdJ/YBhpBo9HS/1IVWKnxGs8d0u7ESz+DpL+Qjr9USk3yiD/x53//47LD7r7ceclpEr49uOt6X5fv9fPW1ku+mxGEGdOVZyUFXFR8LMq94Fz5zO1ASXO7YmjBF+zyK9G2bsKiPfcr27vPmTxAlYlNyykcryTW2hAHxEA4ZSs1Avrfjo/P38weLY9fwioSRD544/hf+FRfMlFA0l/Y3J68fa4LZT9Fdu9TBspsTr1XYlC9Nd3r7k1G7FjOwP6Q+uGsU+DKc7d3Mqx094vT3gPqcOED7tPKAlbibKa8fg8eVqu7N3jGEmXJ9zNmOrdrG0pyy6OxBC+7VRmHgMullkopFJnf+pAxENIB30UkFZzWZjOACd0n6GpqwYPrOgjQBgb4ZHN0E5u8OzqTqtvHb/bZ9PFOoNn/H6XjV9xboEPDB1XOHKbVxHxkL1e30BM3ek9cINWdWdsTk/2LJRV6XZu0uoOE/doBSLqZDBN5NyX8Y9teR9f8TuDKQRV1gU7fbPhmToF+aZoihsirzGa9xGsA6WagEBS3NDp2rZnog1yzMIb1WENU9YsZOShkL1VyibNx2856FmfTuw15VoEHxC4HME1Sy8J+rOT3MYr9yqCWh7eRFAPdidvqRiYgPonXoMDnT6FnuLeSaJIMAut0xlZsHQxZ7DZhSpvaedRLDp1Q8PENzYtF21r3pRXpTphyfSLlJLT7k0T6xSjiE/4gJPxDUxSGqK8bnulMvRpZhtQfGOeh/TwT1Nl4qCrtJrsfKZrB8VUePyY/xSZIlP1WJzaAcCYG+BhnU0xUHYoOEac7pldALfTdn28Da4OfRaHfBlzKprn7SWaNptG3AzfgQ50P/xCAc/49t0jAOxYx69GE4cuDHiILM7hW61P2/zBusXNLh6s8DH55vRvp+vT8uz059M3eAbWPoGLSYN3ApcJ61vr5oZ55tbWEj5OQ6VgoPGd8AqB4728RR61nXKDj+2LK1V9sGgnyTQXk9jOS3JnLmZ1Wr3Z4/nInT5nIN9bwcM2MELdrted9KE//8Y1X8gsJLK+BmhFDZxtgkb/0I8DchZNtA2BXB3M3LAmjJVkREyc04hcpNQhYCFhfJBhghGgvG4kx4DBNa4bxPjcQ5CB2sQxEEZ4nEL6sgq8848KMospTp0LWbGmPA9Nl6Zhugdi8qPAzwxi6KKHk7FrK6f6HnNbCgfvNW5fsEB2rpkZpaz/3OPadvDW6bxz7QfitMbT3Ut6bt6cdbARXRgmVrf9eOSs1nmvQn2aCLiClfvzKWXqFTOlY9t5KnhsXlkxY/AC8V/AmHLPsV8BHSbpm+DiywM+hFHCWnprnP3TqPC6NS7iMuZvRWH6KnR0XW1B/yToAfpvo6t2osuW8HxrXfTL2yioh2PomSs1EZTvD0XRus19EIJKEJgG7LLJPTgFcmkn0DP+bQT75ZWfV/8PW3CWr/t6sD9X/zJzuN8UMvjJFME+9OKqGLxz9gMNzdjXE/itbyGxfqxz5afJrO9M4mhEdQUEC5q5Y+muGecZXSjFq2xewzX/1hP/9oxrW+ZBgdXd8Bs5a8q+xuOAXZGCfQeKAr7rItSjLzqsI9gx58wSbhmg6lQ4rFXsYryh/TC241+KYMH4hnQrSEsHjMTM8sc0LTPyX957RNO0XedC6veW0jsC+pDtA0+OxNFvwc3Ma0/SKZLqh9yBoyHMtkcLlzCep2Kfh3HlLGNCzPokVsmUv4MAoRnWev7FDozsIreYV55+/uO47u/+VIq6mDd1mfHJriov7zczQKaeA2duaGHJzajjlmOW9OP4EGiM3dwn9J2imYXCzATueBk/4+NmH6wj5RnbYyc548EDbslCY1zjp+VurteZ+LRcHOrNOjoRqd2t3YBCV7/B+LlRJNM4Aa/KoR5r3pOkKgvJDkTIhwnJqvfOGeNaOpHfuq/d9NYwrjDCeHL6BUwt9A+8DXJInVsyAy5DWp1BTsg1D8ZpKE1peDZgHv8mnCCCwW/nvspBNQS0KlMKAVbI5Gv2zkjJKyIIP61h+pUDr3gqylVoBKv1eugxZ8473I1mJMTde2bp3XumomsI65WgLYfif29ofvG03NqWipGCfUWRHURyeD7T8xlXMI4rhphQj8pf0wmT2FbswJT0kAnAMB4AH83wJPBm9eELiiRO5LqBoZZsJbbojMWdgZgpOpMxlXnTzYwh8mLFNonHolO/+kjcT+PMdkedY/shE8M4x9kelC3bTnfP0K/vfP1E17m6Z9ldd6wyjThKGcevm0QeYxoPsyP/r+JOfZi5EH6wT0TTvppZuV044QpsM4jDjbtSCazBBEOYUAXG/AIRP6vIQ+t1cR+1TX0fXRO2GrzLGfW3xPyEG59Af1LPqBEZpydGdD8gxqvYIJn0mO4qHGf5Vb6aOiUF8pIfj7wnRxwoOU7fDFDjAruyJPbBJALbEIv92obRx5BlyMZyecQzuFtl3aBegkLcMD8hi23esZRjtm3EKdLxxT7IibfnD+KyrVqm+Y0N1+3yBZXsfDRDO/CVV/x3R8hGfCvwQlSNgjHkZcw2J3Psb8WOZi2JvUHFaNFtJ2r27KZurxMrZHzkRlp4CqytS35ZCsBczs9wriv+JVZAkm1XsyYnta5LfcTovwFkVAUT\", \"config/data.json\": \"eNqNVE1PGzEQvfMrIquHBAEJ4aMtFw5FlSr1gERvkFoT72RjxWsb25uQAv+9Y6+dDwpS9zQePz/Pznue54Nej1UQwGNgV71nWuYEr6SjDBsuoK4VDqW2bRgKKarK+PHo9OuxBffYYji26I6FAu/Rs6OOYCYVcgshoNOR5PBweHiS8QUTwNUYuDCqbSJIt0q9t8UF6EpSSUR/1btnP2GKih31mCrBt3h5DEQJOoIY/eqiSWZOZ/jMmYbHGjU0yOWMN9J7qWviD67FjPXQWPoNWX1QyGY/3nOXFr0fN3HxXZnVTkhRub92prXv8yVAj3jL6Xwmpu5M6wSx30bKG/RBagjS6JzJ27fGhbeAkrt1Jhi6l00S6WTvHz0nDVNDsg60+RoRzFsld4wRHMio1ejk8ygzLEHFf6C7Uv70omhIRexnPGJFmfFofFkyxBewXkeLQBsM79oDK3BYbOLwsZUOOSjFs8s4gpjzUlpUbFOvdWidEVjkzHXjE4GFDLxyprQ/Nb30IeWTG7JrdyVh/d8v9/zhwU8GM9Km38XXshpcf2JH/4L63okXnzQZFHBf2hdLWgw+OlP58FJthfvvg0E2dIpkfHfXy0YqBW4eAgH2pNdtg04KbtpAL5tXYW2j/Ix+EcLZuPRfg+ZSz7g11MGkFLVYgaBOGS+DXJI0uuIaa0gLwkotw5qvZJjzeHqBaFMwM44rWc9DPW0KvRfknyQV00Yj2yjZlbWV0JlVZ1MaIzE9vhyfnp9nFmEaUp5ETy5kf3yotkwNNsatt0xkkDSf8vvdEGf/X4zoK+MiV0v7sOBNq4IkI2EcjGcnBdTAE4clSJotkQ5otDgQmxfxZbSpBNpq9zVNQSxQxzfB/COZeeP5bsWnEMjnsb43Zc3oMm5ogDjjffcQuFmiU2D3R1gB5rHzITxWePB68BcS5t9h\", \"config/data.smoke.json\": \"eNqNVE1PGzEQvfMrolUPCQISQqGUC4dWlSr11t4gtSb2ZDOK1za2l5CS/veOvet8SCD1NvY8Pz+/mfHryWBQKYgQMFZ3g1de9htCkeedaryCutY4JuPaOJYklbJhOrn8fO7AP7UYzx36c6khBAzVWUewII3CQYzoTSI5PR2fXvT4gonga4xCWt02CWRard9KCQlGEUti+rvBQ/UD5qirs0GlS/AlXZ4CWYKOIEW/umjWM+czYuFtI5JGAw0KWoiGQiBTM3/0LfbYAI3jZ5B6R8gun+75mReD71/T4pu264OQo3J/7W3r3uZ72B2bzY4UBMEOZ7kMm04mE07+TYgqOE0HZYseKDk5ufg06RmeQacbyHb7l9fFYQzxeCcgqsw/vSk7zBex3qQCQhut6MTDGjyWInp8asmjAK1F3wMCQS5FkZb83Ol1Hp23EovZvW58YbCkKJS3xZxsSfEh7+da9T2VsznH2eHv7YN4fAyz0YINHHbxPanR/Yde5hFoGLzcBtt6iaMCHpLbOuvj6L0zKsStYs/IZDP/+2Ckhk9xGd/MBmpIa/DLGBmQ8+XJpm3QkxS2jTx3QsWNS+Wv+IkQr6bFfwNGkFkIZ9nBXCm2WINkp2ygSM9cGqOEwRrygrFkKG7EmuJSpNMrRJeDhfVCU72M9bwp9EFy/+RSVcYarHaV7GTtS+jtumtTHvK0fXlzdfuxJ5G24cJzzXMTVn9CVHuiBhvrN3si7o/8efTDteM9aP8yyr1WTsNKNK2OxG2E6dO6uiigBl4EPAPx3Cc24LH3IHfzcLufJWjV4SzNQa7QpImowhO38q7ju5WYQ+QuT/IYcr1XteC7hOXZ9jaEbgqEfUavwR3/LgXY/wjvwpPAk78n/wDAF8Ie\", \"config/orchestration.json\": \"eNp1kT9PwzAQxfd+iigzEfkDEu3I0qVITKzW1T45pvEl2HcVCPHduSQF0YHBkq1793vvyZ+boihP4P2A5oSJcCh3RemEPHn5QGofmrrZbpvbQYCqs54h+J79MVYTpDdBrnzgXo7VuS1vZpgDhoxs8ijJ4j80G6xzY2719cuZMFV2gJwxr6Qpja9o2RDEhXOYI7zoOcwR9o9P1fNldX8dgSF5TRAYE3AYSZebul5G2fboRMvGQMLqtCu6dZLQIrGZJPfGCyT3R3J3v0qEKJA3PULiI4KWZFBWr1VnWdMtsgjvIUo0WiSruwFmjBMviqa+ljB4ArVNqNe0aFbI5U9+GBwijqKGaEdyS6aurevN1+YbINSWcQ==\", \"config/report.json\": \"eNpdj0sOwjAMRPc9RcWaRSifBZexTDBqRJNYiatWoN4dN+Wf5cybzPhe1fWKE52dFRcD2LYPV0hxyKtj3eyNvvWMeByd7706FjgBd1Ego+eOCmjMm6SRO3QBNPFFfH7KLfJvzeZlMSXfC5YhiZhQSrR4EhkuqvSJMkgsC9TcLskTBdt6TFeV7irMEoptIbsblYpmt170AbWEwQml0rSse5qeMGuDpyD/hFFiWi4gOpejm0M1VQ9AamKs\", \"config/train.json\": \"eNqFVdtOGzEQfecrUJ6bNgmlD32DNiBUCohLW6mqLO/u7MaNL4svC1HEv3fGXm8uIPUhijJnZo7nzLGzPjg8HLXW/IXSM80VjD4fji4D1+Mf+LkUzcKfn34f33D7GMCPz4VfhGLczUbvqFCZCuRQJim7KVTC4LkFKxRoz6yRMaHgDqTQwEqjPQZTogOoEJ1NZp/i7wo6Ucb8sg0pRQfFCmMctQqasqeTSWLhVq6Y86ZthW4QqLl0ECGhCi65LoEtuK5kgkfaaEhNa+A+WGB4JhxeGL0LBweMS8m85UIj75ND3NsAW5O33HJF8TXGMBrPiETMr9o4QVNUPrZD0BQksugioIL0opTcuQxLHEVTreWeUibvJ5Npj9H8iHdAXEc5qvgzq6D1CwyOhyAetuKeM/zGkjoqmym4KirO5DS134vOYnS7T0OTe8NcK4XfqSHqQpBis+PjPpb1rC3Pck6HgoI3DQ33HxAeiSaTgLeixMDvJBeTppGGFOv1Y2CtsaM/fX4yzqB9dk+vn19Y4BUJ+HHI92BxToE7K7eWS7MYW5JNJXsSDt7C0BAZGyz32sxbw71G9gV7IyVu0qJ9jXoD7cAWxgm/IjUx9JLuD9Y48BtXthbQqFAlX1TCkjom+DZ494FiWSaUPwi6EbwGpkAZu2L4NtRC7ktgARXgT7FjVmA4QLmActkaobfOgD/AdlymC+ziDc6Soa+ZUCp4XkhICYw43S6pNCXWLwHavZzZwOzAuWSu9eBToXD5CxNsJJ0NpqM3gxWA28RZhQ4+9dr1eDotekc4tmmug5QbzqMNHWgaodo9d2il4RXrxd0DIwn5PLIfD4vwKD09l8iKj2UUbHP0IpRL8Ax0R5u8O2KnD1++ze/zGnHftXjegm9u52cXvzZbbshsPXzy847dzs8vrq8yjsuUBS+X7HXi1/nZycPlfS4YNMCL2aT3dZ0fsw4kVV1cnV0PL1z//8Doj8FU2bQHLwf/ADHq8eY=\", \"config/train.smoke.json\": \"eNqFVclOG0EQvfMVyOc4sY3IIbeQAEIhgFiSSFHU6p4pjzvuZehlwLL491R1T48XkHKwrKlX66s3NeuDw8NR6+xfqAIzXMPo0+HoMnIz/oG/S9kswvnJ9/ENd48RwvhchkUU4242ekeB2taghjBF3o3QGYPnFpzUYAJzViUHwT0oaYBV1gQ0ZkcPUCM6m8w+pucaOlkl/6qN2cVEzYS1nlJFQ97TySRX4U6tmA+2baVpEJhz5SFBUguuuKmALbipVYZHxhrISefAQ3TAsCccXlqzC0cPjCvFguPSYN0nj3hwEbYmb7njmuxrtKE19YiFWFi1aYJG1CGlQ9AKIll2CdBRBVkp7n2BFY5iKNbxQC6T95PJtMdofsQ7oFpHxar5M6uhDQs0jgcjNlvzwBn+Y8g8MVtKcC1qztQ0p9+zzpJ1O09DkwfLfKtk2Imh0kISY7Pj495W+Jw7XuicDgGCNw0N9x8QHqlMKQLByQoNvzNdTNlGWWKs54+Bc9aN/vT+WTgD90U9PX9h4YDXROB08A/gcE6JO6u2lkuzWFeRTBV7kh7ewlAQBRsk91rMW8O9RvYJe8MlbdKhfK1+A+3ACetlWCUBoO0lv0AY5CFsZNk6QKVCnYVRS0f02BjaGPwHso29tksobOEWoqQXg8+BadDWrRieiLlU+0w4QCL4U8pbiBjaqBZQLVsrzVYn+ACu4yq/x2kZZdse5c2k1jFwoSA7MKrpd4sqW2H8EqDd85kNlT14nzW2HuQqNWpgYaNLRWeD9uh0MAG4VJxVmhhyrl2p525RQtKzTfLpZFPxaFMMDA1Q7wkjtsrymvXc7qOpCMkdtgVKlhWjq4lV8WYmwjadiVgtITAwHe3z7oidPHz5dnpf1ohbn8vnLfjm9vTs4tdmyw1proc//7xjt6fnF9dXBcf2lODVkr12/Hp69vnh8r4EDCzg+9nkM7suN60DRVEXV2fXw6HrPxOMvg82HXPKcPBy8A+2VPOS\", \"data.py\": \"eNrdPV132zaW7/4VGD7sIRNZsdNMp9VW6XFipeMZx87E7nTneLQ8tETZrCVSJanErtfz2/d+AQRIUHa63X3YnDNTigQuLi4u7jfgIAg+lOk6KVO1TJOb5CrdrZJFqt4evd09PCzOXu7tf6s+JOUvm7RW1XqZ1ZVaFKU6zq6u6x/evB/u7Jxfp/xFZZVKqiq7ytO5ukyhWaoWaVJv4L+zIv+UllVW5MrqrQ6TOqkA8qyEdvBxuPOx+AxQoEeeQgd1mSyTfJbOB6pKVuslPnxOsTc+AaS8KFfJMvs1nQ+VOk7Kq1Rl+XpTE4yddVnM0qoCdIo8NdMoi8/qqiw2a5XUKlF1tkpVks/V5zKr6zSHOfB8dqt1OssW2UwBfepquBMEwc7OoixWKo4XG5xXHKtstS5KgJPnRU1zqHZ29LvyCnpWqf59NdNP10l1vcwu9c+fqyLXz6ukvtbPRaWf1sukBoKu9O/SAK1+AVTTr8zPutzMasZyViyX6Yxw0mi+LTZ5nZYDNU8XyWZZzzPdeA3jAkq64QdEgz7Ud+ssv9LvD/K7gToCEMnlMh2o98kavw7UWQq0hZUyk883q/Ud0jJfm0kAlRPkEbWem3d3SYkLgi+T1svhWlYMP/5iPlabOlvu7OycfTg+Oo9PDt5PztRYhUFdJlkeDFTwCThiTmuBv+q0qoNo5/3B2V+/fgUN8/Vwk+X116/Cvdt3rX/Rzg+Tk8nHg/PJYXx28P7D8SR+dwT/9/b0+Mf3J9A5iJkP40UG/5fNg26Hj6c/edrDdKj55OTt6SE0Pj54Mzm22y2Ty3QJ/LUDK6OWRTKPYc8ssqsQF2aE66r+i1YlUruvFa7bBbwb4IJMRzsK/nH7GNsDRGxKfSP6+DmDt1aLYbFO8zAogUKwbMUcFnEcbOrF7jdBhPS+hrVapgwY/2ULp3e1WSyy2+EMdu+iWM7DSI1hEkNk5KDp1GAFCOG3IU4sZNiRaZYuq9TtVJd37gtCgdf/LlktnW/p7Sxd1+qIPk+AcUqcALztggAOqVL1EfYAbHpqGgb/OHh/LFhuSuIa2F2/bLIyrdSHO/z67+ovZ6cnIJHSObBzAaCB1WAHAQXnQLw7oBjtFBjSP3VEeYhyNe7MH+gKggNEZ5ZXNYq6kHsNaImjZgqM+t+T5UYj/tbFuQAwq01Vg+gFaaSKy59h7wc8ikxoDrjcB3MWurg3SM7hwxqUAAtLYAR8UWxqEKP4tEpXRXmHT8lmDq0fCOIqo6YAsAKqp/NQDzGcZ4tFWqbNVCIzU+m0ZVKL4L0AdlekEjk2UvcC5EFPDRtUgMfFAohby5gXMrXpRZ6s0mlESgsfQT0oS3BMNWpJfhd+QjzUd2O1R835J7TnISLWNyyhh1k1WxZVGlabVSjfB2p/uDdQyWUV18VyvJ/u7r/cuoAksF400uoFiio9Ib2U66LK6uwTqygYTdWF2g8MSfV03QWcDq/SOgyqGcCGn5H6A+zOHJRgsA2f82vQ4EY3XwKTQO+UMcGJI7gUBXtaglIU1V41uIBUNeQX/pleBCD5qnidljFq0QCWAgm8DQ3uOnT6dchhGBtwyIUKIjyTulhlsxjlTTwHLQRS8A733Uhrq0Zwoh6s6iwn8o8a8XoCpGIUre+ojtK8Hq5u5lkZ8o9qfF5uQA+mt1lVx8UN/WTU6hQFUlLeAW/aUFASxyw/Q/s9v1LPQYzWq3VgiW0DSYT256cKbRK5NgkG0gT7V2i/JNUsy8bvEpDAA1g/EGf1+OWA9nR8k95V1nzwH/ceoqWUhsE/c8GyqIbAfssEtrzB1SFtJEszh21TfOI1Rc1OarQKUSLFQFRbzQ3QIgE7I6eXtCpLoPEFfhOFRyJPNJ0G4UhVbDCkpanCzk58B0OfFPU7sIjmWvhoexQAgcABsafmRVoRLAID4gdhatlD2DcikDQvSg56yFgmD6+WxWUoc4kQszXLD5p7GDkIE8CnIHpSGHOWkQChNLsGAX8vI/2hfFDQHkxoB2PZL9RH1iTLQVyDsbHcrPKQ/wNiVttzuFdgm2hpnM71GuEWgffAECi90nYXWrCmJc8JTQVWQvBFxoosI4Jg6/dESX5GWgpmD67gI4wMvRZIIgDP45AMZIC6qT1YZNs23BPEX4PsFh311gDUCN43YyDlPyfMNAQ2aEbS5MfXQpGyoSFNsyHoo9PSTR+fFeLimVkbG3mBLYU5QAXARojJXRIeqfqZxMcMTptps4/x5+/MFQ4xY9zHPQRNZvUmWZLN4JIULQSHmo7d4MCe2oQWgKiik+Uy5A4N2R0o3DbyLsWFQcLTZWqv0cVUr1CxKWfGHwEzDDxF0I+xcRuI5KCaecDGYYbpO42NCA/++U+09l64IgMgDNHKjS/vgJaheLDDy2Vyk768DC1HnFQTgBHFhJbsFaiCuIKv42+iIf8M4UNwmYFxIhOJyWJbZbfgmZHpBVwEvhq4jGWZ3NEsmp88mV9hDtx0mFTgp6ahce5QZK3vLPVFmhQ+p2UJhjaoL9RD4yC7AszBmmiWA4GGv6r/xP97/dpyF7/ai4AhnjkO5Jt3f/zm1Z++Ptx/O3k1+eObb6PHwbz8UxfMt68O9159++bN/ldf7e/vT950JIYfoX2E9G+KXVvND0jFGFcANAyuF66ULJhL0oGYmtZmJRMadnKVoqCHUbxkhzUi9rFXjMH3LQMZGBFMoUEehxBptckz1OME1oJAyHz9KlIvFJv2L589g5+ieWEVUSTSB57Hxd6UP4KzULDApFbP3Vb7U4evYaDP1+CvhITFd9xnoMCQb39hsGDlD9TLKLIQhQl9o7mYA2Dk7lsrEcr2JJoOFAcDfp/FkGn0LbsM9QULI6hGGiNGQM+PNYEAX5QgoEZqPR+i4fQOfw2Uoyt8VkF7BiBWizybkUAmiMNlMbsYDUhPhA64aKonEgA4cm2GgO8yT8Lgu5OD167IArwwVDREbGMOPsXsFodmTLZ6b7UJPAPJcFWUKKlIdAzrIqZIVjjHYceGTEZqGUgcwWHRNULvQptAI22yAD5ZBahSk62eoXaCa4xo1oogk8lP0btUmxT8fkz6kYEOkSzr0DEqqdW20SardX332FhCU/oMc58tgdPV5BZU0zEHjg8wOlDIbIMgOMyqm93LZHYDsiLFdoriB6oAJkrzFJgLPuy//Gb3El5ylEwdHVakRDk8KzKFwq7siwG94wx2YxyD/FguQLUA36GjKuqOHYdLtIVxC1bNtnGtH6fbF/p1QlkXRNfJ6A6zycGdvgkbKDiFIciUnIMbuJAczNUvQwdAb0cYPZ1t0CP78PHgh/cH6mewCXLgyBWIg/FPB8fB07tWd/nsuizyYlONT04/vn9SZ2fSwduPk4PziTo/eHM8URm6lOCxg4sS3sBWU+eT/zhXJ6fwvx+Pjwf6+516c3z6xnrPGYWjk/PJD5OP1vvAHevDx6P3Bx//of46+QeBbwCCZvzp6PzPpz+eq4+nPx0dNh1/63wm7z/IpIiRY2IzFbozsBB6KgINuwIHYAyleeFwXLuxG0fxb23CdMhsFVt9e8IpBieJWs/KoqpYkQFue24bkczbmgiY+QY+o2itYjTDstxq3+zsZD5nFGVv43qSBTuwuGhkkg8XZIhOB5bq69vuGXYB2YT0hcewAddM+/M16D2F29ylKWE0Yp+lhnnocafoPkwfj1qjHR9zFDG/SsPWIkbdDmbQYbLGCHMY5ultHeo5RAPLx7PC32d1sSbSUCirA3UN4tp5KcqBp9dpflmCUHfe9gqOw8nxBLbHu4+n7+2NEURP6b7CuGtwdHI2+XiuTj+qox9A6Exw35/awMwmi9TfD45/nJyp8PsoEEnvDkT8KBvpSfub9vgZTOLtuXp7+uPJefgs6sxGHZzxYC0BRJ3/cnp0Ygs6aHuTF59zdXrCD0Nk5fH36uDkUF7o+Yx5pY0M8UD/6c8ToAh3I1b/7vX3waDTTos/nLbZEFHkNgRrKYXxYHOEkTGXzRKht/n/mXDj/x26wU4izYYZMBZ4QXdD9UnV52Pm2K0d+uQndMZFc8VBJ5m2RWB7R3/64vv3bbOgLZUsojryLZbw0fd26++7QuW3rt/WCc6K1SqrwTAzughVEdO+EmX0BAfuCXqoYfWxCjlVD9bn7CYMXv/tbwHPovG/8BePJIEo/sGRZUKgpbUtFao50dadNoqtyRJr6Ln6QgWPTw68ZPD0JMFOz08KCkRPo44Qh72cyM3P8WhbaEGzewopyrTaLGshA2+X5DM4BCN1WRTL3sy7SE/cqqhnKTjSa0bZRhIPYfqEqJGtYSnl2Gdr2XAQQFm7OADPBodpDTyOaYEedB5szwtIuYLpZ/iba32C9gg2wp4BOmg+cI/tsMW1vHfN7lVaXxfzYKQC8h5jMWLXZbZKyjvMTrWkgbA87AsbgxjjfMsEQwjg0AK4Hkr0wjLC15G9XnA9croFukMkP4qdZr0Y8tqAJJOFQQgOJ/jG93RyFtft4wfc+Or+XmaMCnred4U+JcDBoQezVRL8cZMNj7MqTjEysX06fYAwl/4lIOyBfyOch+bnQyNTpE4AltQjNDuqiBqb4BKVQeHoknQH5ZAuQQmtk+E7fCKIKJqcWCB8RYFbYZIR53IFLg91pNckVOwmFGfM8qttbXCMNMntJhrHeVmsY3AXKljmUAqR/PkhjjC5CcRt0UIQ0rNrcBFovmf0yBR2ih18af0dIgy7aybbNLBkt++3CPP0Fjcxeaf3VmDNKnVylI+DC5ddaAhMGpkbqJ+LacSJKsnRUtVKmaL9scZcMMEEqbpMrqoxvGej6u3B2eTxMWkozBfFGjgPyB4q8xGtJk6LVzGn4C1O0V54HIh+40BMf8mu4ct4lazbELh1bz/EDFbWpTXCELBSRjJq0oKNZ+1N9rWK0ugraEPhLjcAh2NfcBsEG0iQk98EVgWaAWUvNIyp1/IRuLrZ8g798eUGE5qXd24Rk3c4HMPdA9sHorYvYMt+FmsGaF7ixgcBSHWugAFSzanmJfWBW7yFAgUWdG3CUP6rKxjoGZlNMyuXLlDTKk1KMKwkJwumKm7mqD/TrMGNecj/+YhbibQIGtK8wEq/qgah/UJIwllVU1yiCyYe2rSBifTIX9wEerho9AgqeZHv6pIpXbJBuQQwW6vNek2lI8grut5qJLvLHoX2poNh28nT20gHjYRQTtGHNBkwlgNbKogwB6lC7oeUCIZNlYslmqkCZ0DFLTriLqDZgBEfiXnfI513eu1ocbmo6gtHBQLqejJdtKgr21otTZJiVpRzLUvcISyxUhd1stQR1z3rFUriMhVrfM+IIF3P0yrP0eXJY7X+ZShri8U6oRutX1/fVZQgskK80nUIRm6CUxsCi7Qivlx6kc51t1WWhw4ocXpblIioxqhDSbsaAbnHxaoZNPs1NShinW+dgGM8lOx9g5xFwedjF9VWG4uk2BKgWHY/rZZm2fu2yVlfByMlpQlSqFAXIfIdZl9jjF3fhlHbUrXnhf0dkrVMaBtvtrub3622Wo2P+tdPJylbPXGTsRfU6ixrLh+jzoCacFxygfgB9Swbk5dDhAuIi7rMbrmtLKG1Ss/cPcr1BySHwp4S0otAQ+aCzJhaB9NoCG7YqmEHygjG4AmgK9I7emiG44T5Vy8bOOq5emkj1CRZm6F4LKwoXGHiMF7zqZVmRC8ZnnuwYwsMdEEsDikqUF1JoGkhRc9AhCXWw15druJWl2DaRomcECAdtRQa9CL8rI0CQ+NxUaLQKYfhp6zE0p+Y30vuMFmC+icRhUPwp2HyKcmWmBfBpe6Zyyq5jU27uExW8aJMyPGA2Ti6otmLgdYJcn4H2HC/4cHALj/S7usyzSmKZbN0wNzQ3nENj1hNHU6FRs7v7tiejdIWPVanZkV8HAN9fa+9/c2KshOv+/ctuReG4S6HeTywnO8WqF72HLUZzOqEK88k0gNZjEQfolbrhmuoR5zUsfCF29m0swEIvzZz0KPKB3tJ8ZBEXSDxYRGooCwGDWzo1E8TTId24elzU0gOeRzqB1t1BLP1xmnDgrAoQ3aF9ftVMrvOMAtgsyuyOnQWdcZfHkyNxBlyB1gH9U9YNV2O+koZbHtKcy5VpZMF41TD94VgKZogVdH4n3aYwQAlw8r8ave36+7H7sitzPUGj3o4zh0ZXnZBENld1mmzEFtEPjja0JEzau1CCRwe9+wljLi9VRehliXoR6kJiLM9wstCm1sWwVPv5IvyL6SMiUJGvjpPLxUvaKSptoYIRD+dpDXaVCRu3damWqDb4fXYs8yjbn5ksdxU15xSsUgjb1uU8ROBjox0J7iVIMCVl+TIjpHKoMFmiWQjXSBg+1LhZszVW251DqfykSwaXPTkacd8toEDrgadYUYlaR4Q0yHmMsKWR2pNwwXh2WGjKZi34NrwVEL0zVrz8cwffRkNeYrEdmdLFr5VndDHDGO3X5OXmA3l8KidFdO0+Y37gveuTo3Y21nQ6d/vFrfvW+4DOwX6CEggeQUsGOUcyQvwvxHM7j3DGe19PX/Qp0obb9o6qKLGlhB9YYbwNf0NZWO/y5EgckpJvoCelLnYx27cesZGyo/bSqABp72Yridr4EbG1bG3OGv+tguLp80aoeQrUXJOYC4CHRcBExPPW/NKLMCSSDmueG/QeLCI8IRTR13V0JKy99rN1CttO5bkulXaoKXJYM26GC/OQjlu8kP0xOg/zk2OzjunEkft+oJHxejvJjufIAw9yfSucJr2NWrJnz2dRkDbSNyI601+U4V2RafjNpD1IwkEIqqpCOOAv1uODG11ZP8J8ZpisajSppCtTFdsYipfkEOn5Lla1ZR49bj3jm5sANuBGUynNV+69X1uVRZxJE5GxoOP82bA0DxFLCiw/DmMnoZEw/DqddOsFXGU4VkWsYI0TbVabEksiZAgg9hnfaRouRXud7pMu+aVfBmarOsctiDWk0XbEJXa8n/1wrhJ0/U4oGMAQdQMe8dZDmYqZpNGEjHX9NpiXkK37SDdZNeBouOy5EmauGxvbBVQk2CNe5qyN+Tq6j7Sow2EqHME9onKTvCMZ4srXwyXg72cRndayKFt53DlljOjMgRDRoSD6UA5H0igSIRfR2zax0Y9UESAcxqL2NUnMYb8nbME3UDx1EqG4okxZns8aUbdKHtWMdPzmyZzZvquVkUeW7uGchf20TBpf7E3ZXh0Gs0cQcvleFSTPqyaRBrhJBwuebFx+zSoPb5DWg7DczdpQEe9bHo6X+PmIJ5eCEBVxpWDlyR7cH90BuIwHh1sQdpSojNbxHIXQDBQXEy07ZTDSaGc7J+cjsQhNcjdORggn0CD0nB8Dw2wH+oWfQzCydUZrdA5IPkI3ezmNmEoZWuIY3aI5DxqPDNxhddNjMemjAPrhAJDuJ5UYpcaVm8xQfDMqrkn4nKDR/6tQ7ZyJENP/HOKl/tYp1tteGMqSWid3OH7C3pnBFx570yJ7psAxWW/czPGnMfSpQeDVpLbU5fQ3VK6JqFVhzCwttWg74qFnaaw0NVbchocFhzr4Io8fGbvN/d+DBuZyGJfG3k7/YtrjPY7iF/KToNcvJMjoM01TYR74Jwr10n27fujdbWDWW1WTipZgBjl/HZFRT0iJiUSiKZIK39IcpBDWwPWaoJIpMka2RAudEiX8gAgJwCknaSk4Hfndgfu61j+4LqxNJePfDGNoNu+McQw5ZPuCmHRAGuZYmguNRULON3IVmwYn6JqC66SMPGqPvDM3EKALP8ES1yUd4+lMslCR6e6FWYMfS7rQDXH4e0rOiynEGTz4IkXeVgbIOGzX4CG50SYFxVcoaFk5GP7lErgDs93zsDonTMsLgae1HFH6vYkjZ1jG1vSva2Yw6OJSdcI5apeuizCe2Dbbe7NGftMkC0J5P+DJDLdFODQh3jEXLtm16hq6jomNJLZ7/q1nD7PQRmpgeaK44R9L4ZqoD93XPfOEc7ODVNtg8R/OkesA3Kzz0BggjV6IdnydDVVzzyDmuOqHYD+en1nGAJ1wZhNh6tkHbZPnUadY7E7fmgkt4abNZobIb/SxdroJtKJ28hLGLsa2osuF3sDut0jwi31Gnn709ll3B9bD7A3p5Ktin80NvGQssliPo3ABmO8FgBj9eHeE5jExnX7aW9zTuDLkdaaeOwEVUOO2IjHjc+R97yZ1t9Z7tH89j9xCWgUPunMVU7EcNIVLKAULYRqDHoiBdEV+E5Fy2H9zrUL/hEv/oVVANUCE2BS9wieCK9EnuTenoKcQatz98OX1jH0Ef1i2x2AguRiAz6Wvc3Nkj/KP72jNBcHTjnthqzT211uDKSaTzKUZDebpfGzBjLooAk6pgQOhYFlifQcSuSzokUuO2YBeguE0K9pWYSyH8aKz1P4euvD6EAzA6hnILk7D0yFTeptIObG0D6m095zF2YUMuG3oPWoZGsPKYdlWHw8cRyjzIxUl/WiwFkDQzMzZtX8gCzrUgezRZ63xtgizxvLUQfCJcYr3NXByp+hinw62db//ohYYw4utTXAGFsCU0t74iyDVweEk6hyyNSyo/1FZnahmTbEPGcDXJMtvk5vsSoluNeHwvb2v7598JxN+5JiNF0qZpEPkXJetI4bWBlPXkt9esDofKl4HRv2lQNO9hkjCbblyXJp2cG6gzmQ0JR6rTjoiK4NXy3BKScSL/Qky+nKF/GKLTMkkmr4jkfnDNPx65yv2rF7WjaFjfuSGgjJ7KRQp2b52s3TwejY/II5ZuoNMoMipTi8Tsy5paCm5YXvHtipCVKvi3XoCPiIzD5n6q0baL566Q5j5xrJOLWTjNYdrp7s4m/OMD7mWToj+NN3XXoxyTnrNtWTcRNuO16psmN549iKnHFJW5MdtlmFPmnKVkUYyTFDH0sJ48qpOwOHRmn6o0LEj1jEeNHe21PeL/CJY0auuIrsATCeS3WSr+VmUvqR5co7Kh7L4xtdGPO5HNEbyeS5vRSTN9e4asp096kdsfLQ61EydSOZEmaMYWIxFUmBqZ8m4Nfr+2hJu4zahw/RfsD8jiDtTvktwzEzoht50VEF3FaFjnI6F8judA8TISHs0kfrU08B5AIpSmiPlL6C2qp6dALkIx1utOr3aEvz8iMEaWm5ouRoB3bygySCt16S3Cvgorgo51QDaGKl3dYSY8PjBvxmpL7Qgu7zNR7sujq5UAmVnrkxdqQuplYbNCjW6TxuSqwv7gNDMn7AZDyFQ0lT48ODFXYdyDsr/GpHULmMGBjFHhX8DKDUIqY7TXGe2+YvbddgUc3uAhuMvmN3JBfseijNdhmzTqdRtpI77m1EOq0cyR9L3aVPFerqw55QqTWvjmJ1T5VZinWV5NkipfvWrM2h3/bsCx19w6hYwMfMKEDWKpw0ybnRtsSdUyEK9j4SAS83QiKgKxb0hauaUFVQrYobPD6cgtJrQoBtyHZ4cNQB6S98ptL8lgDvwKWjxCl5vEys1nEHOvNDvL/N/RyobW7joA2S76gHIr05Pvjr5OXl7tev8P4t6xJIIfquCWySbeM5wcBw0NvbxZNKc2PL0t9TkNou2H41/jkINyHRIrQtHrTk7D37jdqrXAHVgFVn1r1gQctzk0V2mzPld48OqYuidBOeCqPzVmyjiuT1H5GmpNPIHsZ/lNvILTeL1D5sQha9PwLUJjiFi5AucsTyixIVD+2RUeEjjzbqv33WBJWnqPVgZGwBKo7V6SyPpaTF6hYLwDcQbCjRwRzg00wgL9sHetISS6+xjJ7+pkjcqLu8ET1UivBIR1MrbySN/NkRf/8ch0nEQonnGYZWLzd8ZB794fITLajb02ZuJ71BZWT02y5PR+/DIrjHKXmMtD2iXkvnjpQXaaS/23Jezn7oT7r2BP+ESQy2CKw0VpHov2oyPMF6gzXY76a6qqJMlGlwUF6B3ZBjVgq+YGXjrMzWiMg4jufFLI4jqyeFVhLpEga7u6yNA/O3SjD+iG9eUL7DwrunPzbbxXKOBgKVjG3rwwZOq5f4MxUNvH1MURi7wHu7LPYUxQPpRGQXC3Ngn4DZlBbiY941bFUOYgMu+2paO39uwv7rIfh9aKdasT4E3+liF39JUqdkx66O4TVuQDhwtyWNnBI4f+unXGrXJfO2++w8U+loejOl9pe2/eMvxBpw53bh1LpEYW8u4a+s4FN7G4qFRDVJbVj9u7atSGMt6XW75s9giBKY9omn5haQpq8rv6YX+hYQAfLQ/MWACPk1wyMtVJkTU4FKHCP3xrHchsWsvPPf2aMc8w==\", \"make_report.py\": \"eNq9PX+P27aS/++n0BPQg9RqvdmmLd4znotL06QXvLYJ0vbdHQxD0Nq0Vy+ypErybra5/e43P0hqKFFeb9K7oE1kkRwOh8OZ4cyQCsPwrbo65MUm+DHfXXc/fPdTsFddk6/bJFDv6yLLy+wqL/LuLuiyq0LB66zcBFlRBNt8d2hUG2ybah9kTZdvs3XXzsIwPDujd2m6PXRQJU2DfF9XTQdNy6rLurwq27Mz867Z1VnTKvN7tzZP/2qr0jwX1W6Xlzvzs2rNE6DYbatmb353al9v88KC6/K9fT4c8g2jtq7KTr3vivzKoKbf7LMy26mGa9VZdy2qvIGfXNDd1YCMef+svEuC50ASJFASvOpUk3VVkwQ/ZTXW4zaHpgBYMxqraQnveOwGw/Kwr++CrA3K2g4Q6A0v4L96Y9+1hy4vGG77rlBZU870tBnQ0VkAf7L1+tBk67u0XVeNSvjdDeC3U2ndqHXewlzIwqusyMq12qS+lusia9t8m69pCtNGYU9ctL2UFWG20qJqW/61z7ruWt22KVRo1pXa8uu+f3gA6qVbgpC2h7qH60VS15dvqnWaHdbmVaw5cH2t1u/qKi87Q5Zfnv7SYZUAZmifr1PksXQDVE+C9jr78utvUmIean2T/2Ga7VSJk6qgtMwAU2b9s7OzH1//8MOLt8HCMOhsp7of4VE1UbjP3ilNpDA+e6ZXCDLKVbZ+B20MzyyXyFqAQdeskuDnqlQrAL1RWxhptiEcI+TFObFgHJx/izw3p5Hf5t01MeqsqlUZqXJdbQCPRXjotud/DWNknGtgoUJxfaYfLMuS1tesqLJNxBVi3Sm+0njDlJUw2Kg5AJXyRiCwydfdEhBOEJUVw15nNa73DQxNNwgugpBBhPjoAJ1h/yE1bKtDs1bQzELIt/Z5pt7nbddGcaAKWDqIQZTSNKVpPINpqIobFcW4shRMtK9H0ZUee09Y7tuMvWo2CvpMidXTMturNtrzMp6b9czDBqZaESUKwA5faSJQGxjKEh8CEE30JglACJYw0KZTGwNxloO0goElwTt1tyiy/dUmC/DdHKFH+LS8XMXxigADSXRzLLzJioOKCT49InQDll4A3Dj4y4LQi5qs3KmoAAYh9OI4FtyQ5UDWf2KbF01TAeMCU6oiNdCQSsGr79tgf2i74EqRqIQFUB207P9DNRUwuKAudaIpChI/394J/k1gZju1q5q7eUCUXOsVMQ9Ga+R/aDUQmfFhbghhm9hRmDfUUd8FzivNZfB9DmKj++Xp20PJrRC5NM3LvEvTqFXFNgF5nBNOgw5JEKGcRs42MjuCyrEtBpy4xqwFoQMTD5QP26dhgPNfdaawVF1RrQcvafVCr3kdhRehmBnv7GzDVyVMcL4BWYbLLPjt7at58AGwuQ97fHA8s6vD+p3qAGend7cOCNht/r6vM0DGVu6aOxcxLRmvqq56agvU+7Wqu+AVlRG+KH/grW9QMBOonTXTEaAgb4GHfj/kKAOQtzNYM0/nFxc00g1NIcwqiDXiPAAshNoO1AQMpGpnqrzJGxBuIIyj8Nl//pK+ffHDq9c/QzMA6Sv//sXLZ7/9+Kut59JoXeQoWRY8WP0zwulNdLckKRYaBeiDuPbMstmmui1J0DKbbVTbgR5BJSok6pDfdlgFYC0kEohyastgqcLqTqurfwFd2vTmS4H5tjqUyK9PxJsGwZKosCBm+klF3xG3LATnJMEb4o7FNvwgmOV+xKQIGaWVhqyYsM/RpCq7Fqi0XA1a4B8Qeji4rmFJF/5D3YWreFStUWDi5TeoHqDFEoWYQCaer2aFh1/FwsSlZoEApuYZ2GDTov70LDsrVlDWlQc1KoSFlBWAkphL0DcGtL+6VlGz/Tvg5Ih/tItfm4NCY5tm8h39HA9DsoDhJtKBkTNfQCAyIyLqLx7DYa74YhFcng0oRCW+ZfoSevm56l5iuRFBP1eBZjoNERRGdctL9YPA6P5iyDj9ojjUNAg2xPTKmNQSnhXCphgz0ZBBP5jG8EgSDdfnfdhLM4USKmvu+vYW3P2s29fnH3CzMMO/vgLb4lq9F63JrGwPe+Te3mgkzSMkR/6HIrFK8jTrAApMML6dlqlyljV5eI67hqEngTPdzjCS4MX7rsmeNbt28SH8SXXZJuuyEHRDyFjCo8H8/t5lDQtoIG2u0UzimfbKB1iyCwcJFyywFtoqtsbSyIQfVbnrrmG1o54kSsG6tNVYfNgRJMGH+5jf6YFQMzOW8cJlvn31WmuWX+3gUGMqvetkSyPYZkDgjQV2QcjADslucwYyRVJnXdV3hjojJKbIZfksGTV5DvB+IXMUJpDbw5w5EEhIzt2Jv4edpqYVGzkggBbh89dv/jt0+3BHQoh8xHzbAXjnmkqPzzNV+ZPn+CUN5hPnlxArBmtytEhH4lgVCnaGH79KtNH0gv5Bm2DUHe8yZ7ewzweLHDXrodhovbavQKv161ePH8XaZ6h5B/2enf37wNOBohh26EARMCr/gA3roey3e0byGp/GsgOxpPR2dcJaX62smU4YMiyUgU2n1S3pCalyjTalzZ1uIfZ2Q13F2tTsC09TWqwRbdW7XAEJ6SVvuQe7Y72d2VcdSnGxebDo9dtv43OaWUnzvbFUI9ZICzDUdtfd7mp/zhN0zjtzOzsTpLDlMW1mD2Xo1tOmRCyQR4ytiXB01Lquo4etCwBqpO3T1G6rHA8AmCslOi42czCJq0JPfGJNd37Njgo/k8w1fbPNHdnVVREN7fJfnqbf/fb8Hy9+RVKBieEpf/P2xctX/xVqO6e9xmWRMmrogyDowDIaWdxcYN/sR9AvJas6AEbuEsskSBbyMOg+2I/wOL8EKOq8ZLdEvMSdxMoFvQw1fuEKekGbcFiup82QnOqZH1wXHV3wUru8Its2CXobE6c132gzvB/MhPsG3krfjRbYDAP3Qnqpo7EVOw4B7duoR4YdIOYyoN6/cwkCBLEVcQ8Jmml+wy42/pt1tb/KS2X5tgWF1LTdpGsBDSoY0ua47+EoE9uu0OlDuyC5HeL+TTcxqUkoWkm+sxD8PGctZj24TfQoM1kg4+mI0IDisd9EoGE61kSGjkB1bnO1SfNyk69VG92lHfDUPCjrWbnJmiYDc7Q87NmPplpyaCWgZ97n+8Ne/2oVSgp4JKz7llZ94FaPAcfB3xe29ZBI0DLrHVy6Aeyyu7taLaAQevjmK2bHNeiDDicKXsOI6GeE7VvqWTcetQXM87IgW2YhRqVZPLtFzyVDvtAPM7A1ojj43CBNNX8/VF2me98WFSglaBtD59hb5GLKdZeRhvtt8CQO/i2IDAjY18e44nkbd3sNUkc30T1/OybXGqRoDtaWsihkXVmV6L4zcL8NLntloVHomy2J0jsAHI3KVsE5UkK+iVfw8giKf/8oFP+uKXwangOUAMkx5oin3RE35Y67Bo7aVHuw7rbZoehSeB8hw2pVAzbfuiN/4JKXMi4zdhrDklDvcbUxT0p+Ee6dqs0pDjYeJ/MgTrGAF4vNZ9Uq9HMBRjP4AcsvstASMrIXwK3kJDZj7eGsgJdxldgWcYxquy4y2HS8zEAxOr4vGuQsq2tVbpBB0QEdMQbCuQAWcCBQ4HJbulvDRqlAUJHRCS1QlAcOQhFFDjmgTH+yFvcYu657LDDyHnRWR7EsDEigJ85GIiOr4Hq7hWNcVdWCPTunwN0Zb39ha74FSBjVmQf1ZvY9bEpeNui47yuQW7x1ZJzGS0Q1BlECDI4kjpqY1DRAENJIZ31UAXE2YYWejaAiTq4exgxL9hXMQVqrJs3JUgdCRHqC9L7M1F4fGjRHnHq407p88uRYSOCNJXAf6jXmhvbEMA4NWtwpQAutdCRLwgkJwUs2ZHXQclxDF4QSxql+MwH01Ca44lIS+syWQLHLb0g9C7L/nYvycltFpk48AynDBiW/evolAeRFnKJSxYFpIsDAuGBW1nc8NlxUGdiJokESCI0kOM+opR5ZWLvAQ9VtWufrd4WzgO9gTVRXXgSwoEcAf3GIPzdSF+PUGFPPOgoqpnu132d17+IQsEEzwrQvwtsvQqE1QbllQAmKqcIb0stiILFjGzC+Wj9fH8p3wEO3hsmdtbUMxTrvqxqnMYpg2mX2wvcJSzu3776lEMhgcNbQJ0pOhvGFqDeG0gu/HiXeyNAq0y8jR7DMcthWLQn6HLtbMRnsSlzAqpFwxbzIVjxHhj/67sf0nwAGxYf2OhrIbwtnWnBzPQGJOZgDfA4D0wrvI3uxW1Fwz6giyyy2KnlbMjQ1LX7+VeIyV+LjIp3YksLSTdtsXxcKechbFVW+4S/+WzcG3DyTq3FckWswchosQ0YwHEygRL1vr21CsdQlLM+yNiU8Htx0/n5QXShbzboq1e8jCQgHDqYBS48EjX1ghhaTLhbhH2230ZJc52UMjOfR6LAXSmOJHmNE77My3wIlAPoHO8Ehx+bDeRBeq2JzXh06InrQ1sB/wrkZ0qhB6WLVjYIFBV2BkM/XgoECJg1yequaG0zgKXHusoINLWRtHCAaMQ5sZIH5EeYQdbVBy2Jp/gje60GAZj1khYGAckc3kZXYomNDGF3EPDVAeor2y5qaP7kueY4l05J8Xl6uRINDC2sNRKnW50YRzwNmD6p478zZg9yYmpoiD2OYfxOZOokL1xEdLtceEzIOjImqKM40xMSInLOR4GOTUZXr633WYNaOfU61tDe2VeISNwkGU28bGoHSQ9IUlObQBXJyXVR3ezTXbNVjJLSVkgFohzRuUdLbWkP6aHt76Uh2R3q78zGguduPyWiapB6b48GEFR5MWteT2Ugmc1A7XtHdWuyuZJFOndNTnHXr61RH7cwGyk6dLQzNDsogqk2B26zZH2pjtph2/LZX8dZW2aushdY4t8M2osjTEGxSgSmYpE8wpGJ6B0P+a3rhwKe3R8377yx/Y3SgwcCJNu7bINN7uxvFPZMzVr2HOsUd9oZ9n2PnF9iL07PWHFucQoyyOrpy3o+j30Knve2mxyRsNFKYCAfhefSMNTtR+w4C6UPLTPioJgwwFA9UyyQiAuODhGjTPSxtDYaeO9hkg6Ak/98q0f8b2xpbYBiYMjZnb/iFtgxqlYF9yeVcMANTu2ruUtpjxDMo81FGkliQh/Fg63VBia8z2BFuWfQrGSUQgzml+qfQ3RDqRLSmTemHJ0xRvsuJ4963xq0RuSiej6iDbjzo6Ml4UAIIdn7ujnbcznDKoJWYOE9fPZOgQ8v8TKZZxsPCydC+112SU5rs7MM+MthhrAlx0DwKFljZ5RRR0W58zi9Mgt/nAU097ob7epEt1qJROydACRjxivLTzC+XAlNhEk+5i8YTPKOkVhWZlFbG78uvPv/8S6mnhN0IpOkqUOGU/DCWwXMjL0n5eaXt3JFj98I+EqpgLkSxNM7qQ9pdYzDF2ICWlzNYQS1HTXCcplYSPIkdM5JogmDT/RXi4lJQ1OzSnltptPXXT2AKyRVEyMFLMTUDKfb1E1BmYf23r09v8rev4/th/8hYp3VuZeaJPdv6o26JWU/o1DD1aV32tQcdFuikXIPx03fGtu2R3oBNvfxhYfWoHIcFuEzBAg6qDrvr+tCZ3QTDo4XtMCjGIqRRcOHKAAHTiBjmPSt/7KKTQ9HGVXqjGtww4o5ldzVLze80lRspNrdEXXN2wV+ftIyoDVLGXxEWHCKqT4LMNLuCXUP5nvb9Pltf56VytkdMA1iV6dUdWO96vWptfZM3tBdjCQvClWrHZg+kPdCH/YGT/oQX2h8QY9v5oSBZ73bqg2OcduG0s8/a3P0zvZd6IHgMpCgYptrX3Z0vwCa8E2jBHlpW39AEoxltFDneGLnx94fopr14tuOPdeBpACM9TnEcJwqI8yR9brGJJ8H+vl1c2pidGPyIbgOfnS1z/FebzSxj058IlwTjaKTrMRw5VSxceL70+vR0l1pNOjgm/Zz1AXQQJopO09TNJ7Cx9knIaoKVKbxBwZFAPLrbOM3YeLbHmBR1Yx8pBx4md20e68Y89Z6q3h537PJHBuv0BvgqLzlDcjxH8TBc5wZ1/xoPAn83NorjwObAqHuqoK+v93rOm56tNYhhNpQhHnGgfXQSnnD9u630Jn/tXRbzRI50dcztPMSArUXnsJaLvfUvrAe5ywJ3BjJxiu1EcJZ5jCVusfTVqxtTzaDhq2WccLoqTq5+5QZg3ba4Tg2SZyOatUfws2thGjkGLggy7du/VajKW3fGnYENJ9oI7OHqQGXxO6jNWXXA3dd4rSQiY0MuuNGimTUwy0XkBLS8POnhQactLKs9Iyc2CUhSMAmaKq1umlBsZcBMKiPLITHl7NifrC05UOuAYgKCWB1A04waCXmlab3Q/w57QP+KLtKZElN97nNA33Z24rqSll7DFJgYfd1I1OrGiwfAMEOfHjbK5iOjrpsTB40IO2N+tBiQDuzR8kjcpejJXOAMBL9G0lymFWnKTkY8TZvy0elHJCSI44fz/oDhn5puoPMMepUr/a1a9Q7DTW4E/8SwE2qezYxyC33RJ6YsB8CgqolV1VUdmZhS7I8pGU1jQhzm5CXFQMyWWxYa1dpvafPSTeL05z069XXqo3HGup2b85ZOi6UIwhSHfYmaIKWDppgYjwnxXhiGErrR0ROb35kIxBjVRPtrffNkeuZjr8AL2y0g5YQazel/Z/bdstBT/5TEC15jzCFCpA9nrl9HUYi1wyRw/UNTigkrs58AOsAfLFhEmU4UlpwffTCzhTJGTkwScPdzanw/hLOk0lRv4E2U9/ZagTQSmHwbwG6GBn0hESSvH7qBxtj1PygZSosbS4ushf7oyDdnf2DmserY2Ig2TVULh6zEVuxbR4hPj2oGzXxUXIJEgL0W4WQC3KzhL3nv1VeNYWd2qWefArmnT7+O+07Pv9hHUl07/fRLzL8ufSwDcDMTS6Vf9yOAS13NywoSLeIFJsGFg6/LDS6u4pfLDxK3x7GFgzvNIxPaO5GispxJHbbuZb5+Esmf2OH8yOG366xO+VSddR6IQBvPweRi703n2zZtFWVIPnFKHsotov7HWUUyj9rvl3CHPuWfeISfYgDQcyQ2u00pUJdfHUwW6TBe4j3O6kL2JCTVdPkBg9ZSe+SQH0H2nroFa6xsed2O0OVN/5Ejt3JgmCHX4REUb238s3S3LKN7EUb9w0DZkeMFOR7OeId8BFmDyahXL5mcGjNYsXvc1n95Wm9uY1jhmM5Bi8btmNlKGJVuDFu/49U8Hvsklk/JtHDLOKPkyco4J2SnJ5Owg/XV1lWrogHboBCCVfdlfAItdW7LfIUWVXQyAfxIeq5h+K2EZcRZ4KBJu5yTiH75j2dvOO1xHnzwIHTvOaNuJd8XzEBXrTtudLngf+eXq5h0GDEvip9Ln3CwEvALPQUOL4ByGdJiBAI3Rp5F6/wcNRrtlswf3FGmMCg+Ls0jvaBgpkU1GXLeFdjk74bXSvxEfnii/3x49tQjN9HljXHMicOjrtpQm8O6TzL0Zx7yJmU59/S18i0xHCIi4JXqwcUF8PGxpEOXIiMF65Plg2Gsjml9e5DQbL308IYwzo6dRH3Lle1eT+fYdVXw2YY4Mci2oDt4WYj5C49ouonZ/MvCyYYTZykwJNSyF39AFJGG6dR0kjF53ONKn5gv6U2Re2gH7UuVG+RH9rtVT6KcrLt0cglXOu4+IKuv1aGr9sD765RYQd/1lSE+CCQczWWfKPKopL7HJfY9MrmPJI2x/o0EkhsAKn+s/W8ApdickwTo9/0QqDYn/VuAHjXaAVj5eCGxHuwBJLr9D3cH4GL3yD2AQJ23AAjDvwOwVeUG4CprwVQoZRTUv7GzCT0OI/oye4ZRN7ef7SUGA/S9c5E5IuFBI9ErfDG4CksKazwGpX2Ji5B9ogldbAXL9YYci4snxgdYA1NMnU9Qzf7Q2evxsKLZRxw7WPaQDtgoPOdEsbXeX0duQdoDrXCaPuBA5sFyZe8cQ9PXYeB7G/wyfp9hjV6oVk2+0xdEuBO11A1Eorcn7WuQsaZpMTCuJgDrs22ClpHBZhgU8oZv/yRGs9bZms9Rj1jNdv/nMlhvhOlJ7ymuAz3s+pYL4Zyx9MSWxmHnaaobKjvxo9HLkZ0nmd4vUUfp6CPRKnKfUIiZsUO9pRuZ6KmCDWGBTLO7zGoHtewHCgVDmEmw2VRbYAOKUJj1/m1wySGJJ7MnD/Z5H3sks+3/cZJ5RF4toMV7v5weNZTimkMSbkRs7HJLhceTYSRHq7PfaC49Sk5iUY9Q34iznlxM5cyhthl3hZ1YPSSDOv2Ni8yLMEM8VnPborxjjYyygaf7Aq8ZoGsFZuv2RtwZRQoXhB68jTx2ILFUyufmFuFns2+24chy0QbLwH8+tF3QosyavCUNKnysNpAQktFj/a0rWBoNzHufCyMceuNG2rkHfI4WqynlI04a0pjnxmAc/hsAi0cY9eaFByEyNbz46POfUEmHQTCpQ4zdHZMPNbeTAX2XYVfVl0/wlMt1tdHOU1xMkagju1/xplJswy/jCZigyVtVtgc2vR/s89tF8HQAKZ1iUN/S65v1bCveab4dAD+VhTX7jlo/xMnOEZuhHj5ish0Nv/Jy1mFWezMv20/De2pOPXXtPXPtuKiPHCc54dZL342X9sZteekl7JwbfSM3Zf2ebxTqfLxBZnC1VTu8BdOxIOlK4ql7fPt0vXyHv5Ppg9g6DJpMHs7+iNPW/T0yhsN7VHz3zAwajSK2AhZ3cJ3j/TF37mbZPVulq8horr7w1mnlIja+GFc3FZYe2VGTlwjLaDfHgIPRFb2u2ThIoR/Mjr3guiVngvck/qx7ryPy2jLGLndXMx00jrg6Xly0wFuA+r7+1BsEJlELNhVQje7BqcoOnUjmcBFUCEROvkw30gfV8Yorj09FHLIH1gbSp3xKvQllLtFDMOw5eS8MoIruhjNvcQB434/JrU1sSi3F+DU4rvsXcwUMpioNifYr1NA002M1Fy8T3AuCSZ53PQxTrOFJ/KhP8jtbJ7hNwhqx2sn3P8Amq4WNaJvtGqX4CrTJlXFHUj1xcoOPpkybHByfQ/URNwLE7vXxib40HnTcZSKOMj9wCX00yAFN7HjMjk+s5NF+Tpow/pQhSteyyWne3Ntj9NBY2n0CS5wj2y8qx3tAezAY1WYw4iDx8EqihJJgmFJor1uyinw0oh8UGkKH20vc5l0mnKsGg8JCQQfM5uK3PUX07skd0MTx2L5cF/SGjy2y9roD65Fmz7DxkROzxJt+dL1fT+hls2g7u21A7KV4S2PkbfVItuyyBq+NpuleOFO/gb09aOqvJlgX7/kbfE9A0kWgfIQo+7ysGk4hyvVtW80Ow9+9Q1QvR86HMNmmg9MKwIV4WsFoxE88O51vBxA8t0nKQ9+9ReA7We3Gh/sa5sCrJwXu9Fy6eIYOgUhvQBZ96CL2YnrkePoYsYfPqH/MQfOPOWxudgtjHKcdT3SJIjCnOMfIfh3cnPET3yxujQhTExPv5MV42hrUvEXH+M33VBwPifkqSp9/6nwmJXKXo5NrO/qyioUx8c2VY8A4b1mKZAblT4N9QDw86JYc92yFvc48Fh9k+T/rk9UJ9WddsX9+XzaN+9O7s2nRD4xuvbY9jT6Vc5QJtGBNPVp9aQpXkw3SXu3Lswq24XRXLm38LcwngGxF82Js1oyJ6AwTraFhrr6wm5aDRP6Vp+kgN3/U2in3ATBJ5+N+qWDlpqhjLZNX77Sw+faeBiKJftjGFvn6ySf6GSHGGzArCqENBhid45YwM0oNDy3b99GJZ5QH1917D45a0Y3+ECuV5yORLpugVDZX5sydE4F9JZ/ax2OwVmEJT7EU8xNWhKxy7OYVWS8ZA3asplHpEa3oPbjyrFcgEtay104rAPmd1irBZPWxRpLs8hMyKn4AxKoYt/VQBWGf3OatUQ6+BlpzjHt6eTnRAoQNwv5PvQh8FaXQdiA/fz4GCuKWcDVm6XMtPQfVXOHqQHVaehEfiUvs8cdqF/xYefqyslL28uy35+dvXz8PiAjnr//5djzdI8Hoa29ocxTEQACK3rV4meh5JGTevD2HpoGRfm4zRzRiL7q2EH3eBl75Z3rKj/WUm54eI/xcUH0JwhlKvcG8O6WrT5R34moHc++AubLAyrOlewPEajm+DmE1CZPvHzgF5vDShJXvCoiHkKRrIk7CkKEdR89CO44bH/A/hhnfJPEwXhrSEax6SMdwGlwlYa9+EKCmb5tYTd8jcRTQ6KqJ1fFLJPoLJMQAj941sZq8QUKAkAW4nGB/rxqF4Sv3WpW+hbx5a/KOFVFfFqyktteqdELRm9KxJ0l60pa62io2fiUJ9ZFeJbep1wQQl5Whpga0PacKtSchCew+33HxOBv8oX0BxlJ3POfVWOsnOWP76w/RgZkS9NMu2HSOEmO74TFfuglBoju4C8EcBjT303lA0On1KRDO2fU+qAQsCw06vFpy5Vphg7H2hunj6eIe6u2BpXgXozZ1Jd7xxM2X20NRkEYKTQ6t08q9L0AfsH3ousyPvglTry8YQop4pYTXnA4ByWU5MqV72ieBN7hnnI9uWSJClS6Pm+8ULya+4to7U+1KEnyYSI4SYYyk9zEnYon2nQvnvzhn7AEp5ygZx0uSwN0wHw2NL4euasdF69l5uEJoRNPPNY3M7Y0K3fQ9Bcne+X+7Z/0Tb1B/dHicrpbH3MJjl8yPpe/gElDnAu+jwtjM4ee6ZyD/RHaDaGlmhj7fCXjt8EO5MHrzWe3Zzyiu62ytHdT0EiPAtsKzZnfADfMbKok2ql03OX2IapGmGxBIaSxa4pU12A01icLzc0DpHFAK+6//6MD/tSrqBWx68HNGztc7MRxKX4Di72Nd8CeSLvRnZI52xp+KOe+qc/r4ZkazsbBD+a6qCpWVrwn7rHhm7sXmVNsFf5XzCPj2qenBflbn0Z2INAd94Q73JSdIz9kebP9o8MkW8x3pq6zN18+ZLQt1o4qFKXn188vXSWDtiihr17ipiNvgM65Jn9/BX/wwhydggTbbwQ9NXkTDfPPV4GQ/ZzX8GhiWz8zSwMyWng15Tu3XdkTwRH+qiTwx6/6T17RMQTlJmL5Pg7FyGn+Gyj20Y9AgaPozQniT4FP9CloOPpN0Nj7kZ9bYRmqI0XobfVdoMPTEO2KRkKtPzlBEK/zB9onHZAZ5PcGhxLwQ+nwbf+20iyyS9L0OI6bOznL8hjHOc5qieg/TFJkqTcO5TmlBDjv7X4VnGBg=\", \"model.py\": \"eNrFG12P2zby3b+C1aGolGodb3IpDr46aJpLi+CSNNik92IYAm3Rtrqy5IrSbrbb/e83M/yWZCe4PlzQJityODOc7yG5URS9KXb79ucf37J/8ZZL0bJNXcm26TZtUVcpW8NYWVSCHXnDD6IVjUwZ/NsUm5TxKmcbXpZrvrmW0yiKJpNtUx9Ylm27tmtElrHicKybFiCruuWIUk4meuw3WVfm57Le7YpqZz4PvN2bn9viIBTaHDjclFxKIQ1eO2QhBMJ70/SdEpY/6kpjOgL+slgbsPdIjibauyOwYcZfVHcpewkb5OsScLzlR5xN2QfxeyeqjbBbqbrD8Y5xyaqjGTqCcGAA/jvmCre8LgVvqqmSnt3C9jKTm7oRWnabvdhcH+uiag3Ahld1VYCcsz2X+8lk8uaXn39+dcUWRmrTnWjfwI+iibOsAi1lWTL58P7N64/ZuxdvX30AyDhqG15UUcqiG14WOakCv1oh2yiZvH738dXVuxdvspe/vPn17Tu1JJP8cCxFti3gryJHcDPU1LdmBIQjSkAxmZAi2EckBFy9550UVygp2Yo8vuoq1MGrpqmbZD5h8AcM5ooXUuSsrkoQ3xasi3GWdw3K25dEo9CAOFklbtm/+W4HAGAHErahDG+Siy3LGsHzDA0rRhXPSbMJu3iOqlREb4t2T/qf1kdRxaDGOgduF1HXbi/+ESWosT3orhQKHv80Aqy5IoOdljXPYwWQaKpaoiJr9dYz8KFtsYvVP3NjOEvwqxQ5WRFL78AcFY1iy2CXGnwZgTVl67qWLYi5q/IIwL9asMvZzOMI5cb+w8tOSTSOfjSO2lvNDp1s2Vow8YlvWhAz4AFtabIAWFq6YJ3lXSbbmrgFsufovUJgZoBZIdm2btZFnosKf2LtXtjg4egZUrm4KTYiWuHGos2xi86R+uihUttpugpshr18/+sQdXFY85KDd2akJdoJkQEvEhED1gzkVnCKUoBZULjzAb9I1pYWM7QoJmrEzCLWSqjB9EAT3yCBbxznEBpDRYDjZBB1lD2hr8nPKGMoIUDBxI1o7hgsB+sifRA+Jo9l0WrqFNUleLshfahzUWZqOFpNlPX/3hWNyDOyIIC9t5xEZGho8RA4QWYs2q3zNkodQL3+DUVwQ5OHrmwLihI+CEVFxNGADwHYbDqbXXrzaNAAcyMkTD71Zw78U5aLY7uHiYtgAsSGqSGDf2HpFuafzHyS/LDOeVZeKnIjM09opo9zh/po60yJsL8W2VlDkAViz55548bOtg1XZjZnl8HCNac4/iUA4nck600pT7LyR18KZiGmAucFaGkD8x+bTqjpB/r7UEjItRBpZaDXa3E3Z/eR+HQE5YkcFpofU1SpFM0NjSo7wfwTw5LkwSLACAAjqV2HJhga0rRoxUHGiV0DntDDh95oEHhMA6Dj+4xbbJ2j2uoFDb1FMbOboi7B4HLYqMP20AuNHkMR7AkEvanL7BYIRUnCFotTUJgfNdTZIKpjMsQDVm9ZSEGFkgCdDeZQoHlRFZOHzwN6TLvHXAiOxmbA6fcLdjZ7+G4/9ZZbgpwda1mgI2PUZWoahAlZXfOBKpcCtIsVCKo7jvATawRjvua77xB2gny2gX3XBzXmSQ/26dBjyAQSOnwBZRqQBZSuGI61NJZ2wSpFIXnYThhMIId7u/zByaFCRAJqLbdtnfXAgSIKbtmtwKIaNyShbBMZiM4bK2TWVTpvEIwqqTSAMvFjI6SoMNhKKAGhfLJkpki+kSqv6H0m1hL0urNO8dokrQubtFxxzzjkLUsMfENjNI5hildgTIRGp2ZgQ8uV48eAf4XCofCfQdFa1pgBdD7IBPIVPZwzz5eUncBkcouxX9YEyMl1fOzJWJ3lCsxoBXUDyhVKOVU2SVt1nePLYZj2lo+UXZGpF8V2q3Ji5pvb6WoxpZpOdz5zMmSsH/Ni0zogxeZnSlFXcjiEGBsuz23yrc3azDaLOpSDoKF3Exy22t7WTGPs1xbIaDxeYPiQqvBV5cEKlqGqPD6TiVeKqxUgzx9cB6hY1G3sj50r4VUZlauJOUrLlxXGn8EctkXDUR23QAdO9sd8iiR/QgdS9RI1RAFMdZxCM9g0/G7l46HQAoAlZGcE9JZnB2UEPhaQx8oT17xnAN4UNYpzKM6bgJwEcz1wb9JvmqikicHbAQVEvbxoVPeUIs4WqJmmdzmwTzDQ3o4sLO6KjLXtIMYtfWEFYpn7tbAieMYi32suVSlLKR00DN13DSw3QFrjMCEayaEhIn3a7FFBBdtlj1EmGNLaZYQNIlhnQlkNh3S2aeXKoQSMgBCob3gbKyKgpV0FzTwUnrn4tMByS/EARq/5oE9bFi1I+3Egv4R9q0b7XXliajaJQdvmBozEBmEyzQsILw1KX/EEDJbdofKShEZwNkn0RAztnSGr0amyCUe8mknthaiW9WY57xnGyvKCoXVk432WXoDjN+ihWvE/6caqbnIo5fJaSDIYKt4w9x2beiOIqSk265ox5ZAgL+JsaY4sVlOo5OnkJs6xfF6APYKPPX2Swh6Pd57yjN8bFCc2B2EQ0MSAZlvW3CH6iZdSBOHLIEw1b9oT111R5ibsSM86sSwijwQLZX9qvzx/ujAaB++8ckofvGFE360PeOxR7tbaODfQVLHXBEGSx1kY7avHP9OJo9dYfJWlRfl88XdoYr5/BlkQvEh3n9iOg1fhSRcgNLWOMrUFbWy4bW32vCq2EJYBzD/mMYsf20LKAPomENjGaQznTcgEZaz0MFRgi0LpOL5B10lc10PfGDJO0AnQaTqmHzIFYI3HbqdZJeFoKJ9T/5BFmQAYOhVoOmdnkm9BSOJQN3cGARRjZO1UN1EIVuPLiIDBT8DFRFNAxvwDyva6yYySo1XfKN4SZmUTQbk9ZBl02lxLtu3Aamxp4ShRbmZdhTyQbPkNL+gsll29ePtPFoXo/RMndlsATtyJBGIVFmDKOrBXyAsoy5tcH4fQ8YrFlAwTtInRgXG40yMd0vCogcKSKWpUNe6ZEykBGwLlA8GR1AjkyDlW8mXHUzZ2o8BNnyvpBAiEiR1J/th2ILmTmWmNo+TLqh30g4fPlzwOjFpElU8q5h1Qu80YmksCW5n4qD+tN4TFSmpjAyZuzPsruz7ML0THnF+ECRQYVplcVPB/vNSEqXkkxk1b6/G9SiCFkG0kOrv28zr0sjsRgwXGgcsnLhlbbryzjs+kwe1I/ADOQPig5XuD5AHyo5cdLR2dDe/NwIPfv59VDiZ42ElPR2SUGASdEtTxGPRUEsKF08Zo+z3Ym7Xee1r1QAeYdNCAjst3jYBd0in+uXhveo+zvVbKRjSj6zshsobfskV4MtsLqQaMUhUEUZXpk2HXgfawW091Uo6H9q7vZ4zFL7T5uWHfkBeBVbvjPrTeXd3QXZEGWSxxrc/mwnypdcmJXujzHHsXSQO2w7lG6Ip0EQgl7SMd29xf2JTfxH2BAvAmbCh/Pfp/2oJu2Y78Du+dwrP3gEg0ZyeMwsJRkYuAJxKZnl95S0cYH0EwCrUKDtU9D4P1wbd/HK3L46B2dboKJH9KD0NLXgyHHLhvIwv/Y6BZubBVu51SVqKNZdwW5ClnDYSwGBGJi2ML9U9/mDr5RXg9bE4Eh8x43X9/TWhliTFA1ZQc+Kaps+1lpk7e4tGzKHNJvly6xG8Or1TrT1UB9UUphdSVLhyV7FTpFGZN/+wHmFFd9JYJLK4xRYMVYr2BbwrmzCfrn9t4hw8DDlw6Amte83UB6aageg+wcUnYfCpJT+8hoKaKiUE1mHECrASNZXjX4dME5osD3idchjmyz1i4CGxxz4+BpFJ2cZlMP54mRCswZbuqBKzFPxD83Cl5MEsWFv1a2cLH3fExJzqmqN6PsPLQr+MVRovv/jSfD+HCxFMnEaYyDDXU7A78UxwQTxn/VMjFpVtEjzCww9fvMTTN1GFLQ39XNguIbkTDd2IRkadAFfCHAIfJi5sCXycsZkn/HUFkXCrS9hgTwUQ1YRM/Ehpjt68rXkNzRKHsSmyIvn1K8eJ4FFV+QU8pbDNlXufoZw/QWNWwl5EnFvr5DtXH2hWfzehTde9ZYQjDDJU5ziMz6H2KNsucZUBDsXURaA8+DS2hPugMTy29NKGfc2RFTuccXowGBYJTWQ4o7HiB1L8/nmsPdzWR3WJmjuh76z2IfV1fz71gRg+ARrlGDldafd4ecuhOsJeCahNKb4yxmiH2J733cJB4WayuCrIWBJTp3RNvA2AjGgmiOC0IQFkcuoNBBbvpGjkQCD7ZyLRSD0XVtSKE6b1NMeqcai2C+vVP4bTTHt0NmY8QqK9JfcrfH07CVYGC0UPJZ4LR3ooRpWtSIzOn16I54NuIcCQEH2ocbzwGg+GiE8qHlSdmxmXdMwe9xROzyYCFobFY4Y7O9jCMGJJdPzLXVyqHWgsC0hbW4CHiFH/OqM8TTTwGjFCwp8MRBawf9k2r+jY2b/umXbtJpoWsgSw0vbEuGn6Q+PJwA+XLvs5d0NIXibEOsbhLyKeQvTJ0eO8Ow/5Ant+7elMBYPBU7H4SpjM6L8QDvuVslTxk7vMSPiPtf2rkySoJF0NfjhPYlo/zasEf/JiMoR9iMoViJqobVw6Fzg0S/BI9INj/oAEbZLumURfaaKLAzpT68aked2YaJ26Rf5oCRYAfhvCO5NKvcgyB0WOUE0fW2whEcYHHXsWuqzvvftMLsa4SsScrqTtNudd0zSlK/44cOHaWBrse16BfImBaDxo9yu+WITAWTTINIVzMBZBeSO6BWj2qGGHgQy87uQaqDFgR2EMPtv+iaxjF+8iplwvfB8yNHNVRyxhI0kOD1+9nkXid4HlMPjX1guAMO/qJwUlmRhEMWBnHoumYgnGcCzM7ysKJpT79U+ut0UEeAC/JpdY7hYuLXhzvrfWSplscvJdTJGS2LSrzbMIZN/ZCoyVD6kU75zT6YZnzvOW1uBu5j1HuZa5aMG7A/JS8QGKJHBuZ6pfSGT7zwksRjUFS8Fz5gadP/AuizrA8zwt17wL8CXxjZZAanc3ZfZ+OH3H8yDjl1AjEaqtnsq7R5GdSbeDoE1eUwxDErkPRmjg3UgkV6n4WZUZSpIRhp0EBz0+udcpFw7CUwu7zbDHlEQ8W0W8taCu7OF9QPTcV1WkyyWTYf5JFY+H4GVv2es+9yLuSMp1Z9PWZUnbBZm4t1GYlPmnAJ78L2rKij/uMPTWBM3iyDM4iHH0AUqvxNsDDHFo1VlQdulmE1zF3dCPYCLwujhCdwgAtsmBxRMtzGg9Ypemo6Sp1JxzgHwYPV1mO1OiunkgDV0g1o0mIXXnGcixErcJyXw8Hq//GXn3alF3ud8+PPzxl8q7a7BswXX1vqX6fZI83l2VZ3+K9sXkt3cNn7eEbSZ05LafHn4iCbzYdZAhOhyu4xwtzka7caegQX1pa/+Xy2kfklWDG3Mda78FRjz49CboYpenqOD0IXsVLKotHktEqqI19xS/nmpfVqqd7OtEo6821pGar3U8h4JbxeHf6+OQ+hkhBdgVeXedZW7fk/cOdPTrRBX87ZvCPNJ8DSp3EwwnbmjkHwUMufDd+rsF7xJ5+N5vZuDfSwj1i383ObU/jswRTT4x9ITw+weyIUtQvNU2LalsPTxipVLjC4HVxOUObpqdx/GbnhLj4evp0K31Bkvj0cI+x7HI2g5nLYOaafqvI7m/xdR6lo5wMFJuO6C/t00xHZDjEP3Cq00HYlRcnfuVqG33QqVDFYIYPKewvWjnz83uY/wJCa1t1\", \"train.py\": \"eNq1G2uP2zbyu38FT8CiUmsrG6TtFU5UIE2TXg5NGyTp4YrFQpAleq2uHq5Ibdbd2/9+M8OHSEne3dwhQZDY5HA4M5w36SAIPnRZ2TC542xbXvOCPT49XXVt3xTsxdvf2M/lxU7+9MMbtskEr8qGs4+l3LGOi77ONhVn+Y7nl/u2bKSIF4sPu1Kwui16mIEh3siybbKqOrC8bSRsJFjTsjqT+6qVVblhZb1vOylY2zEckmVzAaAFjxdBECwW266tWZpue9l3PE01OMuappUZ4haLhRnrLvZZJ7j5/odoG/O5ai8uALP52grzaV9lctt2tfkudr0sK/vtYAFlWVvMfV8WirIikxxnDF3mu5rdZ3I38Mjewlc1IQ975FOPP28OS/Ym2+OY5abp6/2BZSCuvaU1awoYgL/7wo4Jj97Limddo8U2nIzZ6YUdeZM12QXvliyTbV3mKQorLWDPJcuzpm3KPKvSXSY0wXCivDJYwgWDP68l7+gE3vG87QrARcOkTcDI26wX/B3/s+dC8kLNbfqyKlKQEaiSFGqszvKuTbeP05rLrszV4FVWlSjKVGpsKWjPtrxYLqLFYvHzrz/99PIdS8ypxhdc/gwfeRemaZPVoCfR4u3z396//DF9+e/XH9IXv/74EsD//g0sLvgW1mWFQq3xhnhSayZkx/5DxxSx1fesKHN5BmNLPKHzNVGm4FOEB4wISmsjmiTLcCDids+bkDegz0BmEvRyu/ouiPAEd3CWFVc48U+59RaKfgvGGOcgqG1bFWHEkoQFMZ5SMCwaCAJacC5GzkKFO7JgvBLcXyS7gz9AJKjTPWR15c3x65zvJXtN0y+7DiwVGIDRKQoQqeDsXd+gDRBoGPz+/M3PmspeKQw4jz/7EjwIe3vA2afsn+9//YU1nBfkHfg1nA0rOAivAOEdQGKkhLDlPOtIciyyLU8n/INcwVOwUoDnkVmT81BrEp1uNLCgSP9XVvWGcKPJY+JbwFf3QrINBz/E2s0fPJeB2k5zVgBRNxZ1sO9ahCHdDJYsIGuy3/j1nncgr0amXVvRkABR4P8FvypzGBlQgVtIN20rEBZcNK3PuuqQCtmSA8GRst5kFTKbkij06JZn5ETBjQMxwIqLF4w1BTetraJrP4qBUPCqWU3ftenix8G5KIKFIIzw8Qn+q01Tb3GrTL0EoAYPTIAm8SI00oqLcrvlHR+OJ7KnpxfdcVDb4I1GLOdP7JIfxJrdaEy3+qiO+RhDgT5PEFmj8WnvoXlNC56hbA3Ja+PCB5dBXmQLOinBrfzSNtoK6+y6rOEcd23fCZAGQWgsZ1aU52eBBxicK5LwoNMNuAU4yrpsesnvxDEDbjChdD1SniXs1BG04h3pprFeYLwH9YFdCtzTX/wle/Lt6Wl8ylazJH7JvoVJs+8I12jjiSlqfuI5xMYUK4CBLAYs0kD74vMOFP1TXLfgGjDUgXv9akSSPmurJJTvwCwkHTzU5wAfZw59ybq+ScuCwsmSKeOhUKpHrB2CCdWZOyMhheFyjYnTgnQHPuiwg9YmfK+itgnWLCRSMAqGZjAyVESOjTukjFa5M5FHs7t+hvARnjmIaJZjF69iOy1NQoFIgXMX8QRkyVaPI0CNcGoyilxfs83Kqu/ING7A/sH8g3YjeHfFUWLmo/a+uaRR8/GWgYKh11iycIA0sxHsqc8jBnpqAdoDGm0A2d+SAZFRd0PNnV7shZOuCdgkb2vIBUrMsSmxkJhaw6kCLwaf8WV534H3lMCsldtZoAcdoZ17MfEUzc6sfJa4onwwnQY1u9GIbpH2tpeiLDg7jeMbhfHWNz8NrI2sA6VQHphimggnlJNJGPsYDfuGYnExl6FhCUDT+GQLK5oBw7O7nZIjh6zqIBgcKFXCJAaLKRODQCUUEYO0fGHYDbU4UhNgSsz7gDnwDWt3wdb4w/TG1Btx034MI/CP3Ra/hl+c/H5SnxSrk3+cvDl5/0V0m95g0RLjP18D4I5fn62/O78NzJ68uSq7tqEsBDLxDCN9WGHtd7Gp0yveCZI3uSnMQeQOORbDCcwlyzWv2+4AJ6GqlBi2kD3UFWo89KTgpksHuQPm9KZgmKZIi/2Z0HNuagqNpqx4N7PKTHnLxizCuvGQA63rKwdYj8SpGUtTP1vbHxzoZn8EUJV2LsvFMUjNlceg/uBxlu97D6Zrc9AaUNxI1dp6HOqvHSYxnlDaC6r/AAVIDbJ5RKSO0A6FGij50PXcO4jdQTx89asMihOPbGs2qaNnASla6Ix4AaSVsB/ErHRzwPxGASsti2nSxgat7h+7EoN6bwpAnSqqyI7DRdmtqbxbOmXfXLSnaRUw75l2Ir0KUcfyAJqtsg2k3rVCOMIM3J37OLCWEKi6goxQz87br0otZL+v+BlxSHz69S2wD2arBcEemWMJRjBxfQn/QgWMvlSQKmCgBCrS9lJphpWordbQU/hJtp2N+z16tHAuy9HExMiqc/bj6mnNAtOpQqQSBu+qn9bY7nLR+cXUmpGCOgAztRVs2UCSHMylSrbWmgGaqbnWjGQ4RbTtMoPncewSvMmo0HoIAP8TJt0pcl0Ud0HoOBefnj4+mjA63+5NCmdGZ9YY71DxJvQ0ORqdWF5lQpBhI6hnGi6oE8QA9GExTaWQirezwHU55zajHNRU930cK3mkVFTrLzVoXJufXeCW1c6ScSsuHBAv7e7RPCgZlcI4JPADvEk1NBIPwvjEvIVYte84WnMxCMyMaHfoe0dyJeR1PrsPAfJKpGLYDbCfnSs3CBENFQdTc6hmgGIV7bA5SPLVvQtgr91CDmAHRVaDGwRdasotF9KOeyqmRp2kWEBBmXPMa7RsgDvc3u3mYYat4GJiBsoEv1+mEspXQM0vrXyFzsgk2O9ME8mgXyHtLOtkuQUjxwzbdEXYjdrD5Nf4pwBGykal5SPN84hUre4Yjx1lEipMS3f9gFRJPwaB8KYIJyBAaNvJ1AqGGqNpinjTNIqhVGmrK0gyYnXMjjaQAdFqxxI0ujs40WumdjfmyqNsOYPZ1S7D3zEo2w5CYGM3qOFbAIMMhJbBuYS6z0M99vVM2906FNMnOJZ4UKxCHHhJQCOQ4MhUe0tBieKPoB2vOhsWCYAUWFDGCZll12UHHfcBYaUU0WsCqy5/dsk1E6YbzK+g4sEGiOKQPNdi6CzPzIa+liuWlt6gZsof9PiamVIc+ROQQVabLL9MqqzeFBlTvi0Hii4g9VubE4jFoclTYz+hkvtyBO0EkmjcA0MvtHB64S/pP8yoLKS6lIg/qojqCyF4heIx0oJlj6ymUGuAF0+ZI0GVmqSQmMTyGu+6CveSTdeLGLaAmWA57tWnZbNtEz+NmDBEWY1WYEo/QihRQV3MHVr8C4bhfZbzUR/KJHHTKxREELtpHThBGrMhhTwwmm8pyD0OHdEB9ZntMZ+fBZOVAfr8eaTelnV2rXsJKXZNTEU9v/F9i/zm5HwvYLWC1erWVKxw9cqsNu3JfStKWV7xIJpw7Hd852gY+D4CoNJ9fR0G+W+Zv1CnYjer+BWvkguI6lJ2oQZdou+xXWPTswc6CBoyIEjI91gxO8aBVWMmk+AkzESOzYZIsJOQFmB4oW/qwxo+gRYJMMFIaEWNpq7HXNHqxAwvl6qLzeKBV09zV06v8a6nqizK75OvISV+9g1TrWN7T+BeLBHCtpf7HkuDVpogRlLX46BmkevZAWbi2geJO8gS5/MgyeEaKHGPwbkdOnfkPrgAbXGJWeBcxZwP8OLJGE48Gc3zBlveRdpCLtyVBSevYe0BCkQ0csf1pLJNxRO6TyQbco9UOVW6FVA+V8f8VE0oOeqetBvs3BVqJHShbIalj+NhXkKt1S4Z/WEYvCUo9Lj8GvxmdWDAwPC64UeFjzJJnuU7pSOPdP8fXTYGISb2VWnu+jY9XjMCYf61tk2Yl8z1hroHqLqShotxNXruqVdctB8bOgPv9sEVD85yV+oEPoUrtwYUXODI7447oYDt1M6Cj5EUSint9pt8TgJOezSlVAHM3KTjRgBjb5Dwz31lhi0wBj95d7dmnGkwUzGp84lVpTP6qipSM3ZntTocdexVBZPVVLku0XfNt+4i14VS8oHlir4KDad1UhRfVO0mDL7U9ceofnhIbmNRLabvAEhJluaAdUiHrPYuvbjjOmyE1tDxiVLXZjIjs0G78P5Ydp7ABhdF9USVCQ2LCZQTcT2P8A55QIcAmNiJUKEAm8QVl2golu8TvGY3DM1cHExuDo7eYxj+6DmHk1TohBgPAbTnB/UtVIEAq5jkExmO3OQGHY7eIJ5QBKEdb6z0zeNMlvNcCN4hpI6t73lXghb8BRKilJVpalnRcpVd6Yde1s9CHjsIUzinYTU4u/IrCKs8uk4wUpuu5JLcAipdL+y6gG5jUjAzXc0EUxQ2A0/NkYP0j9dyo50HfbBuxxLrq7c4CzByQOT1vIgdnlH1I9xNvB2xaqgPlAZOmKJgfVwgM9v7RqJs4yO+O9OXXBZ1Jo8ZyVTYVHWcKlMZ7rdQ5O5t1+JBh9s3jXrScuQSz8mU9CW9H7fwTvhWkYL1jwO+a9vL0Cu57R53BsIlU0RS4BtefQz21LQQyKFandA2l95pF+qOOG8uHJsZIOaUI9ES892zlXYyfJytzpPZKl2LIRmJxQdyvH0yEwA82JkokDw0LI8vdxM1sJwEpF4k6r+5aDxi5Wz1+NzNqY3wqQabDk+sVl1hGGxptgXq3JOygWR64D4hD3Z3mrV7Hf2AcPL+j2LfVfmXKcYueIPreFo2OYQ1wGRdolh4K++CDKcPBOfaQZ+xk+Of9PHuzX0dHOrivB44HN4qq/4No2P23OFT9/2tSg2ESqCCKfv3Hd10xZFOz2y3Z0ZrdTtTvdwFxZ685g3HJUAyMfV7fMm91uldPyXqBZvnLs5GN1RHymF8Z96BZBO3tHKL4rPAgOjM7Bgm9P/J6PvSaayrB3+pfTiWHHkL6GA/0rBJJs0f9dDpWAMomoodzL5zxTvNNl0i6CWcWUov4pIHvzt09p4+wEs+8emh2zQwVo+R7cxoo0qJ8Fa3LUb1stuhMvd4CtB526RXfu8m2HYj0+HHPBuwpUNPNlTrEvVfNNOo8rN01TadiXt+yBtFLOqYgh83QGpA9w582FFzIPmE8sJHRE4HNxXJ2ey2JjMduh1m6nwOExW6CSSwiARvy4Z1wfk4uqNdjt74h3iJO1dSj/3cUPMlw0cf5JLz/fCC16QuU59oFSCxn8a5gI4O8z9gmPQdvSz5RDxlDb+WxjbZx7Kq1G9k5tJkQLSkm2wT6dlX7PHEa49/waB8tqpO3KL8vthhHyGOVtqyD98DedTMFISzxaB/1MODecUACA3fLba9PFIKrm2qeTOi7db2JZIbl7LbYHRmap2+ZP/f6sn/s5b09OCF7R6on1LZ30x5OgBSWENRBYgG8t1r1E8qSj9PQXqnaBafpQj1blhP9d0UXUal2DtWTyGn91Nr89pCkGe2AM+7ix5ztbc0gzfWeVdS1pekadHm+BuhYWWcFQVuQ0vwOkc3zvA2fJv1lUx0K+0R+RrdkbsLgXd3v8Ku9IALG6J3LlbXBaNVgRoVj0Ds4u7NAWKFeebDdzxyfYVHdNjzhJ7fPhiZujlYOV2qlWxX9MMQ9Ugpwdygw59e9Nx/DKsxuueuVQGDn1KCoX+tvNL7A+h5/fK6lKEKyu7qCNeDVppfhtEPqdIUsaVpYH6QgagX/wUtyJyk\", \"viz.py\": \"eNrlXHtz2ziS/9+fAsetrSEzNCM5ceK4VqnyOM5u6vIqx7s1eyodCxYhiWuK5JCUY43X3/26Gw+CD9my45mdufFMKSQINBqN7l93AyAdx/mb4FEiypJVC1HG5W4B92t2GZcrnsQ/8yrO0pLNsgKfs/fxfFH99YcP7JyXIolTETiOs7MzK7IlC8PZqloVIgxZvMyzomI8TbNKUtjZUWVLXuVJViXx+c5OfR2sSuE6R/O545ma86m++leZpVb7hb4uF6sqTmTvOZQDId31Z6xGD6p1HqdzXX6Urn12zJOEnyfCZx94jk999kX8tBLpVPTwGeRrvGK8ZHlS6efpapmvsSzNdVHO0wgKsF4k+7aITLMkK0rNxvts/jErlrJWeZEIXqTBUlRFPDV1+Grqs7wQ07gECYZwAWyH01VxCYwX2VRe7uzsHH96/+k0/Hz0/uTs7ISNmLvD4M/502Dwcu+HPceHyzf7+yeDAV0OBq9OXj6jy+Pjl6+OXtLlyYtXb7GCarr/4ofnJ69Uffyjy5dHzzSV/TfPjl79oCocwH/Q1Nt5/+7jSfjl7J/vT74gH84uVtiVvwH+HsL8fjg6/e+TU1khw8ISf/4Xf97gzyX+fMafH/HnL/jzGn+eQOOjH999Cd9++ngWfnn3PzjY4WDn7N3Z+5Nm4RB6+TE8/vvpP07Cz5/efTzD7vZhHDtHRRXP+LRCHTjn0wso1+owHqPW+KysionPPmapmIB0IzFjIRpFiGroop4dknp5bPc16tMhyexrXC1ICYMsF6kLupRFoFkjZ1XNdg8cD9ViAQqSCFkf/woB9pKSegdJxiNXVvB0r9MsrcRV5RarNIziwuq2WuXAbxRPqzFw6yMbwHISl1WrUJfKMVEx/cRpNZGMIHHoaBbPQRTWQFWn7Clz5GMHL+vaAdaCGUEaC+giK9YbCSjNJgqqrt18Ka1wm/5hokQSqgY2jWnCyzJM+VKUQGeMF4RaeOEzgKSUlWBVInJ147gSy9L1fHYh1qOEL88jzrDsEIXj4tV4OPG8CZGfxSlPQigsCM+giyW/cl2sCYaZFdHYMQ+diUddywfYsxoz9AXzyldJNRpIrpUGuLVKGPn6pky1rgussdaFMK1u3XrsLLMIJIWVgJ9N1YK5qFya1DgCE1MyD7CVZzVqjV4+MGpaxVUiXPQIh1K5qGt1LWmrmxYdkjQo5Go2i6+oCsjVcUjD4UbqZ15k8wL9Ez5i8aw7FQgCAyaSEmbcYf9mgIuFSKu6yui61ebGIdJgWwUHutTqWrJxQ33Ia0nTceypmjnXONIbbEHjpCs5Srxs9TQC1q71EG6uqUfoXUmurNaJCPlVXLr4c4gOJji6gmll8wKldp5lCTB4VqwESQVBSYoF6wdVPL0Ic17wpSQwcs6zagETSWZSxj+LURMwpdahclIN1E0iBGoQXiE12dD1apCikqCECoVy5u7zfa/n8YKDJmF8oCwSxEiDMFVlT1Dk4ngQmFJBEhhJH8GTfMFHg+C50SysIYUElhaJK6UwSw6oq2XzlsMkkXCayHdoz9q1YcIhR+wcsobXHBN59meWAHY3nniT2gwcwzG0t3xdo7VV7k1QCMit1CS7iRwQe/q0p0uvj5TFxpIXFwLHoFxpo39V1u67XXWvQ/BSFGugOQiGe52+UJXg2bNgryWNr3EECnfIhsGzffnoRk8exH7xbE0OEzRbB1jkY8EVTXkl5oBqChmmyh0fso6D/jdpfUv9YWymSQ1ToNfYHaq17NY8IthUDYilmgOjbSW/FCGAIsSwrgL9uTTJt1ToG39pnLEsQriU46DbCoMJaBgFb3jF3xbogXZsBjYOEtQZ/1G1y8tQab2t6hBnoSjIq5MsDzWvwCKipOU4VaEEMB1eNmpoz2zTCJYX8BSkhChajqSxCjDeKswu6NazCW5bPcc4O49mPo2M5mlk2H6KwIpyvAmgnuP3Pohm8EAPw3oA9MwAABMBgsKEr7NV5XqmGCcX/nWJiyiPR88GA5+dn2dXIOQppD4jp7LAq9EEee6pSbzwCOZ4dO0cQ+SC4AhTjpZC08icD1lkFdx4tX4EVRYC366WBYZkMNcjM+sgAogJqxCUGtKIkfPn4MVMMYcqOU0ySJmAPVk0n2KGkYipHrM2P1eLHYIPow+1vbUqG26wttaNdm2FqePuhJp4Ocq+piVfQpDqPuFFwdeAAGkeQH6EN1YIWxf6LAgCpcyAY3NSDwQ02X48mBi3oh7/ZcTaUX4ntqZeXOiFl0THveQJ+lKECrokF0g9KPIpuBGyI2i0SmNALWwOWFfmfCpcUBrV/S4b+h0GQLcg4RQjaAKu6sVzryGyDdyMVaeTfrakTDGRDClRhOBV5n5lMzPwtwaYXgQJfRNp4jUFNzp+83sC4HZ6Yik3PLXRz9Xxr5LFT6u4EBFUstxyHT1jRgjBEua8FJQ6EDLFabiEsDkOk2wOik8ZI8ipXViTs9uIogCP32hhilQ9Pi2ycDY0lcy9cmkEdzEwRVmKSiT0QIIons1Egd7NlbYNprhapqVnNFa1tdSTx+CU/4FTfYK8uLNGYgRh1fSiZHq6d2m6YQ5EEoElXStyNzr3gbxxLvMe0wFmJpKZOMmm0u1PmklKw2/KuAC0rkBa7pDUXFLw6powlB6i1nRN2H+NOlXQVFrViKJOrQBG+BXxj8hWrs5R10vk4Rk5Agpk3eFLn+0He4qbnKcQ15jVDvxz76EpAM1YsHtLgQws/Q3kb1OqoynMF5+uW9fkCnvobVQ+4Amvd9+2r23W6pAe42ufuZIqqKCPUCIvSFo+W1O0jq4Ggr3Kw+n+Oc5dThmHlKgV+UtCBEco6GEwgImkuR2bPijKlORknNl+bqghL720FJMbKOmnzSwCFUQqZ0Ol/QbPKg0aOWdYCGJ78sTOKAaWXt9OtGbdkATLjSONV026Q6/HsnxWQydIXaSrJd4KV9mu1wxViR9+dYlk3TrvZpS8jJw/vaA/p5tDmZh8BCI23J6KEvpzpITRICFnVhkzuYOmHDCZkzm9UpsZAD3ZYGulrafdFfXoOj9kWVnRuquF64ZOX1LaICMV1dX6ul3DRMxFGrmm8qv6eTvTtoK8VS7HqobsvFeYy47JxTp3eULvLvmQ49ZuUfuHfO22n44VHnCNGBNjKN1KNuJMOoRQYe8g04YtRQSzJvTtdi5EAK2cPOBQKwYBAdV026Gi8mkY6fbnHfaKoAnjGwGyRULHyDIP2hwfm3DWbrtNRPuExu/bbE8a4VcRlpABRCtQlN9Z6DUeN0MsM4uIQs6k4Yrjjis2Tvgg2Le98B1gLMtaXRlUMpZG5Zvg+RGgpQ0rnZ63a92PE7unQIN9UWrx7XDRD1VKQ2+1y1o5HSV7W88tNTbMwFiWj6vJ1vr1r67SzayhHmQpgHZEMR7IZ3qRZxAcm1Kl+qoxaj6OycXFxAC35y7EutTabUewgE9U0VN6qjb4IKlRte6xxGjHCap1M0rQ3Hk3W9rpcIhm+sK2oXNe3GalXXlNdLwxlmMbUwzUSlJ7JAPtRDQXKlY5x1SmEZkMgj2DAT3GrHMaM1NF9rXUrnOs+uubSPaaDSaWyU55BUTdFqXW6DtP+yjjkjMug45oh1IHYWZntBzt7ZsRHZv27BwyoAs1HrDbgs9Rl8ltuZulHiwFT0Gt1EqHysFktDYIBgoOlhAFyGwUN6Ig1MM9kN0O2kjd5jOh6RsG7L0iNXsOJGlALV6uluEiWxUl7mQ9Yc9eSNK3NQOLzMNzAaohwmWcriqhGr8YNMwrTIWIKPfHjfxgKuLErQfzxIjpaYNpkoV+xNOoOaLXOpYd/GIO44uBkIe4incG55B3S0HOAH7v9BhQ4ECMH4PAQHC1sLRIR9ct4d44dzoZO16mlU0Itng6XYBeuxArgn0OgQao78hZ5bkoQBFnlT36g290Vk0XdIe/kvudEaBtEZ+vaAvqkaMv+N/eVH2ou1ryNJ7BXG2zky2XR0PdxN7LVoBXL+ZcO2WexJVzyOhfdGTILdzL7W2gu0rxMa76aIpj1QisU46NKoFZjql8Qnu/sj3EWjeNnJVqILyrRQq1NlGnvA4YeOV4Zosd61oStJZ3el04jlCO9UqutHK57IR4Z5FR7ovcBtQDx7G/pf9DUBzKtdoGQYCYQfByD5Rb+0bb9epR1473jvFbabtZ25DZCy5+ad8opwGTbjN9NF8YQ4R0kMhtpbLoq6/Y98w162fIOcnBZ3opQt1Kv0OUAwk4nvZRPQHIFs55vw2C4EkRxpJs7rRjctwzLt2rhvlsiNuPSWXvjbxkJiVzoXdGjHgPRGHqn72xUOTbg/Xb1h0ejI1duNuMj6E6Y4dAAtWvOucIZLG9++KzFFPmBFhVJwxae6uknNJDQ9OgXPBcjFVkhfuu9Pg1G+63d1ukpwe60Phlo6py0K8kIiwKUS6yJDKxCDLHU7RZ2ScEP0/ZntzAljwg0UYERAdssq/1ijX15TU3g+U6/OY6xm7NaMdA01ftJo16iPZ0VuSaWhwGz2fyrEgtTckhnrBBHJZ7O153XY/8huwCD/JBf1gC+RIfOVMBbgUXkS/tG6Na+GMC0K8LUFRiQY7htSVZeXJFmnfTl4IHWlGoTgOe9mwi6RpNpXkEF/ttzpX0SbEmlRL3GshlSNiRaK8cxy1bLc6xJqMmnUlqchunZHQoc+NBs4J/bXuzmi1e4u6fa3b/9IqVGrnaFxpZMEnnBqG+KIoS7RgSz8s4EiMnnoNiYWQUp+R5TIk1tFr16qX1PmbIyoAdsCpLhKulPDQ0xCNwIo/iZWlt1/fRlnYaSq/ldmrgDKfgQcDv5lkZpzN5nYq5um5Tbguyh+Jd8lNwg9hBa1O1CpCr3z/wtowYpGkROWx4cKB375YybSPTBREtsq99fE5BVSCdxRu0XsiBRgP8l1+hfHmZi2k1cviqyqwTBmTI6OupE2RwVHuKXmj3WU/f9aS1HXMdVSmzUEbYdrb3qq0d+udC4CoJTOP0Qa4d2b5n06ZHP82+7n6slak26w8ksMdfi9M7b/LIhUgpRozuWC9vIe5VWM+gY09nDbF+ffan3yBzSvx7DvroHMPujw7UdChEs/tQiGaSgjxxT/sXs7heBL+LgDVkdbynORjv0YjTEaHmOFt7As2eO5VvOSVjJjwA34QR3120wPF+G/YseXlB+gXAu+SBvA3xbY1Q/LTiSe15MIfbCFeyncaoy7iIIbhUchqpVxFcgqyhgixKn+iUuOXcsNDzEBUeEc6sEVi29QfDMQgp2uDFXJgY9kUnPL8KkGm9vheaQTyEJ8f511vwy0ROIVgHLzdvBXaQx+wJQmO9I2jo6Oigxkhj5XWdrfb9jAj8msvGzp9536YTLq/DCrpvxsrrMC+y899Y/IxrLoe974jgWpNMdvB0toQb+8AbIMHeYDDUmITHUyB4lPV+FkVWhkl8IVxsbNfReeRg2z2LAQZwLw32pWt1AlDFc6/ZC5P6yRxZrdvoBah62Ua2qANlOUnhOYiKXlOxz+DJZx4uz1hUvWYsf9A4A9UgFwBwutS8VYxw2cw1Yd6alRpPO2c4zcFlmM44XQlrOKhf4VStnZmBYOn40LfHMbFOJFIa8GzPOp6SgwpU9KMzRzJOrexug1nf9FtTKKeQkqC7WU1dTa1+GqExUTFdwTPUTutwaLdJvaPcbazXuvTBX+ZeU/+Ui3udHeSGjqA6Wck4GoOGu+v2aip0CjdyaeIK8aOyCtZUoMVliivvhhQTugJBVfpoVZ+ELTZqY/qepjLGhD8nS/JZVzi2XUGD4c5Gxarnyu+dZb9nbnY2KmLbfDBfxfgjyFa4w9Y1Jr+1pqv8cMeoggI8TWJ62azVPVrcaPttmryMaRo26vI99Xg7Hf6Ane7qXS332mLCKHRnq3Hzea+9GuW7iu0s7c6cX03HH08vOzpJi4PGHA5bNiKnqDavp3bltvlZ8y4tz9Dow6VWFWtCeWtC+eYJ7T3AF/RN6G2Tyr9pUuVqTj2lej7bMmgr9RhDAvAr+l89puf01xzTYetMYr3zf8rTKFuyv64gm3E2ROYUpLLPWRlX8aVgp/c5rdOIzh9Gohmlf0rF7mW5e4r7e6efjrc9H4jbt/T+uISKcnSNgKhw8tD/PWzW1sEvHvbr7ONt2LXNiz9CxPz/KayF5KkVwwKzv/UY1nwhwWfyEwmtEKD/Cwp3R7bkfTRFujGUWs5fVzLPN0az/eQ0JMroc/sYNudRJKKwMVoUJWAAIIXtUyXSbxn9SiabfsTw+sAQoSMiv8u9NbJbw4ZbJ7yHrn/bVP6eY95fQvO/WevvpfHNGPhB8e7DjGDbKPk/bAu/sh10bKATkJ1Kedw3BvtsRPbtoZehtSuZ+QPFYSaiulcYJgp1gE6tgrbDMVOh9cr6Y8RfDz/5nRWRwBTOcBfgW5dqy9MdOyWIGAoccxZugtsSU5gb/OrOWL6Fjr8TAE2BKkVOFFApy60F43Xn+Bl1rOxvIfBFb3ny7Pk9Tl7j0t8LuRctydG2zvMDzzpzls1mwJXe01aIqBCQTt65u7J7zOvoBUB69W8A84xrshYM4eeSrBtUQtc0LbTF1ra7Z5+OOOd0aF2fOlu4a/a94Y2YH6vTMD7TRCV42wy3TppR2fYnzbRIoH+/PlmOWInctdjoe1OO1AeXYSXwqne8v8fDQoMDH99+QqEAMfrQCYS08EzfyUG5dOIIMeLapiFP+Ww6jXPQv/0EKD0GOmCU4+9IN7+b3EBoPrqmrwVhqVLe7yYIPJ5Dgw99fZ6JhoufJyrQrl1v0gHjeEnL8sFgfwNOf5nKsyP3g2l5Qs5V3D30nN1nUexKSm+HIFSjmnBtVPC3eOyuBkoyN5qFDZg6E5y+dCe/1Mbxxe9N3wbZ+DGQ1kdDSN9CqeNWsZxQq+B+3xChd/tsqu1Kvwh2V1luzsHaqG2P0UZruRcZLASP3GcDb6s2NYRvm4FboAwMSkiGkFpBMkmKju9m+dgWG72XbBfQIf5GAb4GA0M279M23rlZYG9jR6kMvXAD9/awoOgK6I0kC/2IOtgGTVFbcxryXj8uXG3zPm0HFt4q1h8EB6jm4IDzBPcSnRAdEXO8+qDyf+htOZnrbjhUq3eD8XsvhVgC7gNPMAZQSyt6ktb0GO/MZRnGGFjSWn9S29PG4T3p/wgItLcG4ber2y+t3l61/WJgX+1JW0hyytShmRYOWjKyk2rsrLHWZ+fC/Q/7T6XKFHBDAKvOw2iIlm8oQyX10U+5nGe3mNwXYu83nfaBsV9vTvsO/MoW9cmbjY3tYw/UyJpA356wW2hY68APJtHNYYiUKb9t/D0vDz2GFdDudy7og1rWd2y6oUE4xxdIDpnrzNWbJGcZAApTdwQL/q0E9Ms/rnyVRG5iIaEv9MKKvm1TAuksV/KTfRY1ooOvGIaRAHTjpbA/LsLswrKK6ko2ZTwv3RME1aT5eRliJSINBQwKwNfhi9J/O/pMi8HWuw0W4zcmP5IILf2/z7TnIh9JHy4h2euPeNY5Qecw06aATU5rBx7oy6ET7SBavdcHnCQfDZg2Pe/8Hz1vMbc=\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(zlib.decompress(base64.b64decode(encoded_content)))
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_B64 = ''
if PRESIGNED_CONFIG_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(base64.b64decode(PRESIGNED_CONFIG_B64))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
print(f"LightGBM={lgb.__version__}; device=CPU; GPU/TPU disabled")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet-per-classes")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
subprocess.run(data_command, cwd=SOURCE_DIR, check=True)


In [ ]:
train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
]

if os.environ.get("RUN_ID"):
    train_command.extend(["--run-id", os.environ["RUN_ID"]])
result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
if result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(result.returncode, train_command)
if result.returncode == 75:
    print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
else:
    print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
